---

# PART II — FORECASTING UPGRADE (Sections U1–U15)

Everything from this point on is **additive**. No cell, output, figure, result
or variable above this line is modified, reordered or re-run. The original
notebook remains the reproducibility record of the published RE-FUSED-Alpha
results; this part builds a **separate, isolated pipeline** on the *same*
dataset, the *same* six targets and the *same* chronological test period
(2024-01-01 → 2025-04-22) and asks one question:

> Is there a genuinely better forecasting method than `PatchTST + MCAG`
> (avg MAPE 28.20 %, headline MASE 3.4748) — one that also beats persistence?

**Ground rules held throughout Part II**

| | |
|---|---|
| Test period | identical to the original `RL_TEST_DATA.csv` window |
| Tuning / model selection / ensemble weights / seed choice | **validation split only**, never test |
| Splits | strictly chronological, never random |
| Feature admissibility | value dated ≤ forecast origin, or a calendar attribute of the target date |
| Reporting | MAPE is **never** restated as "accuracy"; if persistence wins, it is reported as winning |
| Outputs | written to `results/forecasting_upgrade/`, never overwriting existing results |

Sections are self-contained and sequential: each method family gets one
section that carries its own features, training, evaluation and plots.

## Section U1 — New Data / Leakage Audit

A new pipeline is only worth building if it is provably clean. This section
rebuilds the panel from the raw CSVs in an isolated namespace and *measures*
(rather than asserts) every property the later sections depend on: ordering,
duplication, calendar completeness, split chronology, target validity,
missingness, and the provenance of every feature family.

It closes with a **mechanical leakage proof**: every observation from the test
boundary onwards is replaced with noise, the features are rebuilt, and the
pre-boundary feature values must come back bit-identical.

In [ ]:
# ─── Cell U1.1 — Isolated pipeline: load, reconstruct targets, split ─────────
import sys as _sys, pathlib as _pl, time as _time, json as _json, warnings as _w
_w.filterwarnings('ignore')

for _p in (_pl.Path.cwd(), _pl.Path.cwd().parent, _pl.Path.cwd().parent.parent):
    if (_p / 'refused_upgrade').exists():
        if str(_p) not in _sys.path:
            _sys.path.insert(0, str(_p))
        break

import numpy as _np, pandas as _pd
import matplotlib.pyplot as _plt
from refused_upgrade import panel as UP, features as UF, evaluation as UE, audit as UA

_pd.set_option('display.width', 200)
_pd.set_option('display.max_columns', 60)

U_PATHS = UP.repo_paths()
U_OUT = U_PATHS['upgrade']

_t0 = _time.perf_counter()
U_DF, U_SPLITS, U_CAL = UP.load_panel(U_PATHS)
U_SCORER = UE.Scorer.from_panel(U_DF)
U_REG = UE.Registry(out_dir=U_OUT)

print("=" * 78)
print("  U1 — ISOLATED FORECASTING PIPELINE")
print("=" * 78)
print(f"  built in {_time.perf_counter()-_t0:.1f}s      output root: {U_OUT}")
print(f"  panel      : {U_DF.shape[0]:,} state-day rows x {U_DF.shape[1]} cols, "
      f"{U_DF[UP.STATE].nunique()} states")
print(f"  targets ({len(UP.TARGETS)}) : {UP.TARGETS}")
print(f"  headline   : {UP.OBSERVABLE_TARGETS}  (observable; per LEAKAGE_AUDIT.md the")
print(f"               other four are derived policy overlays, reported but not headline)")
print(f"  horizons   : h = {UP.HORIZONS} days ahead  (matches the original PRED=3)")
print()
print(U_SPLITS.describe().to_string(index=False))

# Verify the reconstructed derived targets against the ALREADY-PUBLISHED numbers
# in results/tables/results_summary.csv. This proves Part II forecasts exactly
# the same series the original pipeline forecast.
_te = U_DF[U_DF['_split'] == 'test']
_published = {'mean_base_lmp': 9347.1, 'mean_palmp': 12875.8,
              'mean_pa_lmp_t': 12875.5, 'palmp_premium_pct': 37.75}
_got = {'mean_base_lmp': round(float(_te['avg_market_price'].mean()), 1),
        'mean_palmp': round(float(_te['palmp'].mean()), 1),
        'mean_pa_lmp_t': round(float(_te['pa_lmp_t'].mean()), 1),
        'palmp_premium_pct': round(100 * (_te['palmp'].mean() / _te['avg_market_price'].mean() - 1), 2)}
print(f"\n  Target-reconstruction check vs published results_summary.csv:")
for _k, _v in _published.items():
    _ok = abs(_got[_k] - _v) < 0.06 * max(1.0, abs(_v) * 0.001)
    print(f"    {_k:<22} published={_v:>10}   reconstructed={_got[_k]:>10}   "
          f"{'MATCH' if abs(_got[_k]-_v) <= 0.15 else 'MISMATCH'}")

In [ ]:
# ─── Cell U1.2 — Split, chronology, duplication, calendar-gap audit ─────────
print("=" * 78)
print("  U1.2 — SPLIT & CHRONOLOGY AUDIT")
print("=" * 78)
_split_tbl = UA.split_audit(U_DF, U_SPLITS)
print(_split_tbl.to_string(index=False))

print("\n  Chronology assertions:")
_chrono = UA.chronology_checks(U_DF, U_SPLITS)
for _, _r in _chrono.iterrows():
    print(f"    [{'PASS' if _r['passed'] else 'FAIL'}]  {_r['check']}")
assert _chrono['passed'].all(), "chronology audit failed — refusing to continue"

print("\n  Calendar completeness (why lags must be date-aware, not positional):")
_gaps = UA.calendar_gap_report(U_DF)
_g = (_gaps.groupby('split')
      .agg(states=('state_name', 'nunique'),
           observed=('observed_days', 'sum'),
           calendar=('calendar_days', 'sum'),
           missing_per_state=('missing_days', 'mean'),
           max_gap_days=('max_gap_days', 'max'))
      .reset_index())
print(_g.to_string(index=False))
print(f"\n    Every state is missing {int(_gaps[_gaps.split=='train']['missing_days'].mean())} "
      f"calendar days in TRAIN and {int(_gaps[_gaps.split=='test']['missing_days'].mean())} in TEST,")
print(f"    including a {int(_gaps['max_gap_days'].max())}-day gap (2020-03-19 -> 2020-06-01, COVID).")
print( "    => a POSITIONAL shift(1) is not 'yesterday'. Part II shifts on a complete")
print( "       daily calendar per state, so lag_k is exactly k calendar days or NaN.")

_gaps.to_csv(U_OUT / 'tables' / 'u1_calendar_gaps.csv', index=False)
_split_tbl.to_csv(U_OUT / 'tables' / 'u1_split_audit.csv', index=False)

In [ ]:
# ─── Cell U1.3 — Missing values, imputation, target validity ────────────────
print("=" * 78)
print("  U1.3 — MISSING VALUES / IMPUTATION / TARGET VALIDITY")
print("=" * 78)
_miss = UA.missing_value_report(U_DF)
print(_miss[['split', 'rows', 'cols_with_any_nan', 'total_nan_cells',
             'any_imputed_rate']].to_string(index=False))
print("\n  Per-source imputation rates:")
print(_miss[[c for c in _miss.columns if c.endswith('_imputed_rate')] + ['split']]
      .set_index('split').to_string())
print("\n  NOTE: the audit-disclosed 87.3% any-imputed rate is a property of the")
print("        SOURCE data and is inherited unchanged by Part II. It is a shared")
print("        limitation of every model compared here, not a differentiator.")

print("\n  Target validity (MAPE/MASE safety):")
_tv = UA.target_validity(U_DF)
print(_tv[_tv.split == 'test'][['target', 'n', 'nan', 'zeros', 'negative',
                                'min', 'median', 'max', 'MAPE_safe']].to_string(index=False))
print("\n    All six targets are strictly positive on TEST except log_carbon, which is")
print("    exactly 0 where carbon_proxy_tons = 0. Textbook MAPE excludes those points;")
print("    MASE / RMSSE / sMAPE are unaffected. This is why MASE is the primary metric.")
_tv.to_csv(U_OUT / 'tables' / 'u1_target_validity.csv', index=False)
_miss.to_csv(U_OUT / 'tables' / 'u1_missing_report.csv', index=False)

In [ ]:
# ─── Cell U1.4 — Feature provenance & availability at prediction time ───────
print("=" * 78)
print("  U1.4 — FEATURE PROVENANCE / LEAKAGE TABLE")
print("=" * 78)
print("  Admissibility rule: to predict target date D at horizon h, a feature may")
print("  use only (a) values dated <= D-h (the forecast origin), or (b) deterministic")
print("  calendar attributes of D.\n")

_prov = UA.provenance_table(U_DF)
for _, _r in _prov.iterrows():
    _flag = 'OK ' if _r['available_at_origin'] else 'NO '
    print(f"  [{_flag}] {_r['feature_family']}")
    print(f"         cols  : {_r['example_columns']}")
    print(f"         built : {_r['construction']}")
    print(f"         action: {_r['action']}")
_prov.to_csv(U_OUT / 'tables' / 'u1_feature_provenance.csv', index=False)

print("\n" + "-" * 78)
print("  Measured provenance of the CSV's shipped lag / rolling columns:")
print("-" * 78)
_lagprov = UA.csv_lag_provenance(U_DF)
print(_lagprov.to_string(index=False))
print("\n    gen_lag_1d matches a POSITIONAL shift, not a calendar shift.")
print("    gen_rmean_7d is a trailing mean that INCLUDES day t.")
print("    Both are therefore rebuilt from scratch in U3 rather than reused.")
_lagprov.to_csv(U_OUT / 'tables' / 'u1_csv_lag_provenance.csv', index=False)

In [ ]:
# ─── Cell U1.5 — What Part II changes vs the original pipeline ──────────────
print("=" * 78)
print("  U1.5 — ORIGINAL PIPELINE vs UPGRADE  (so the gain is attributed correctly)")
print("=" * 78)
_notes = UA.original_pipeline_notes(U_DF)
for _i, _r in _notes.iterrows():
    print(f"\n  [{_i+1}] {_r['aspect'].upper()}")
    print(f"      original : {_r['original']}")
    print(f"      upgrade  : {_r['upgrade']}")
    print(f"      matters  : {_r['why_it_matters']}")
_notes.to_csv(U_OUT / 'tables' / 'u1_pipeline_differences.csv', index=False)

print("\n" + "=" * 78)
print("  The single most important line above is [2]: the original FEATS list")
print("  (Cell 3.1) contains NO lag of total_generation_mwh. The deep models were")
print("  asked to predict the level of a series they were never shown. That, not")
print("  the architecture, is the leading explanation for MASE 3.47 vs 0.49.")
print("=" * 78)

In [ ]:
# ─── Cell U1.6 — Regime definitions for the Section U12 breakdown ───────────
print("=" * 78)
print("  U1.6 — REGIME DEFINITIONS (train-only thresholds)")
print("=" * 78)
print("  The shipped stratifiers cannot be used on TEST:")
print(f"    coal_critical : mean over TEST rows = "
      f"{U_DF.loc[U_DF.split=='test','coal_critical'].mean():.4f}  (identically zero)")
print(f"    regime        : TEST values = "
      f"{list(U_DF.loc[U_DF.split=='test','regime'].unique())}  (constant)")
print("  A constant column cannot stratify test error, so Part II defines six")
print("  overlapping regimes from PER-STATE quantiles of the TRAIN(fit) block only.\n")

_reg_rows = []
for _c in UP.REGIME_COLS:
    _reg_rows.append({'regime': UP.REGIME_LABELS[_c], 'column': _c,
                      'train': round(float(U_DF.loc[U_DF.split == 'train', _c].mean()), 4),
                      'valid': round(float(U_DF.loc[U_DF.split == 'valid', _c].mean()), 4),
                      'test': round(float(U_DF.loc[U_DF.split == 'test', _c].mean()), 4),
                      'test_rows': int(U_DF.loc[U_DF.split == 'test', _c].sum())})
_reg_tbl = _pd.DataFrame(_reg_rows)
print(_reg_tbl.to_string(index=False))
print(f"\n  thresholds: {U_DF.attrs.get('regime_thresholds')}")
print("\n  Note the High-price row: 85.6% of TEST sits above the per-state TRAIN q90.")
print("  That is not a coding artefact — the mean market price roughly doubles between")
print(f"  TRAIN ({U_DF[U_DF.split=='train']['avg_market_price'].mean():.0f}) and "
      f"TEST ({U_DF[U_DF.split=='test']['avg_market_price'].mean():.0f}). Part II reports")
print("  this as a genuine covariate shift; it is a headline limitation, not a bug.")
_reg_tbl.to_csv(U_OUT / 'tables' / 'u1_regime_coverage.csv', index=False)

U_REGIME_MAP = U_DF[[UP.DATE, UP.STATE] + UP.REGIME_COLS].copy()

In [ ]:
# ─── Cell U1.7 — MECHANICAL LEAKAGE PROOF (future-perturbation test) ───────
print("=" * 78)
print("  U1.7 — FUTURE-PERTURBATION LEAKAGE PROOF")
print("=" * 78)
print("  Method: replace EVERY numeric observation dated >= 2024-01-01 with noise,")
print("  rebuild the feature matrix, and compare feature values for target dates")
print("  BEFORE that cutoff against the clean build. Any column that changes is")
print("  reading data at or after the cutoff. Zero tolerance: exact match required.\n")

_leak_rows = []
for _tgt in UP.TARGETS:
    for _h in UP.HORIZONS:
        _r = UF.future_perturbation_test(U_CAL, _tgt, _h, _pd.Timestamp('2024-01-01'))
        _leak_rows.append({'target': _tgt, 'horizon': _h,
                           'rows_compared': _r['n_rows_compared'],
                           'cols_compared': _r['n_cols_compared'],
                           'offending_cols': len(_r['offenders']),
                           'passed': _r['passed']})
        if not _r['passed']:
            print(f"    !! {_tgt} h={_h} OFFENDERS: {_r['offenders'][:8]}")
_leak = _pd.DataFrame(_leak_rows)
print(_leak.to_string(index=False))
_all_pass = bool(_leak['passed'].all())
print(f"\n  RESULT: {int(_leak['passed'].sum())}/{len(_leak)} configurations pass "
      f"({int(_leak['rows_compared'].sum()):,} row-comparisons over "
      f"{int(_leak['cols_compared'].max())} features).")
assert _all_pass, "LEAKAGE DETECTED — refusing to continue"
print("  => No feature used anywhere in Part II can read the future.")
_leak.to_csv(U_OUT / 'tables' / 'u1_leakage_proof.csv', index=False)

# Horizon semantics: at h, lag_1 must be exactly y(D-h).
_f1 = UF.build_features(U_CAL, 'total_generation_mwh', 1)
_w = U_CAL.wide('total_generation_mwh')
_t1 = _w.shift(1).stack(future_stack=True).rename('truth').reset_index()
_t1.columns = [UP.DATE, UP.STATE, 'truth']
_m1 = _f1[[UP.DATE, UP.STATE, 'lag_1']].merge(_t1, on=[UP.DATE, UP.STATE], how='left')
print(f"\n  Horizon semantics check: at h=1, lag_1 == y(D-1) on "
      f"{_np.isclose(_m1['lag_1'], _m1['truth'], equal_nan=True).mean()*100:.2f}% of rows")
print("  => the h=1 lag_1 feature IS the persistence forecast, so the ML models and")
print("     the naive baseline are anchored on identical information.")
print(f"\n  Feature count: {len(UF.feature_columns(_f1))} admissible features per (target, horizon).")
print("=" * 78)

---
## Section U2 — Statistical Baselines

This section decides what "good" actually means on this dataset. Everything in
Sections U3–U9 is judged against the numbers produced here.

Five families, all rolling-origin, all scored on the identical
(date, state, target, horizon) grid:

| Baseline | Forecast at origin T for T+h |
|---|---|
| Persistence | `y(T)` — the random walk; at h=1 this is the classic naive anchor |
| Seasonal naive | `y(T + h − 7·⌈h/7⌉)` — last week's value |
| Rolling mean | mean of the trailing *w* days ending at T, *w* chosen on validation |
| Drift | random walk plus a trailing 28-day slope |
| ETS | Holt-Winters, coefficients fitted on TRAIN, recursion rolled forward |
| ARIMA | SARIMAX, order chosen on validation, exact h-step from the Kalman filter |

**Protocol.** Every tunable choice (rolling window, ETS configuration, ARIMA
order) is made by fitting on the TRAIN(fit) block and scoring on VALIDATION.
The selected specification is then refitted on the full original train period
(fit + validation) and applied once to TEST. The test split is never consulted
during selection.

In [ ]:
# ─── Cell U2.1 — Evaluation grid + trivial baselines ────────────────────────
from refused_upgrade import classical as UC

print("=" * 78)
print("  U2.1 — EVALUATION GRID & TRIVIAL BASELINES")
print("=" * 78)

# One canonical grid per (target, horizon), reused by EVERY later section.
# A row qualifies only where the realised target AND the persistence forecast
# both exist — the benchmark defines the grid, so no learned model can be
# scored on points where the benchmark is undefined.
U_GRID = {}
for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        U_GRID[(_t, _h)] = UC.eval_grid(U_CAL, _t, _h, split='test')
U_GRID_VAL = {}
for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        U_GRID_VAL[(_t, _h)] = UC.eval_grid(U_CAL, _t, _h, split='valid')

_gsz = _pd.DataFrame([{'target': t, 'h': h, 'test_rows': len(U_GRID[(t, h)]),
                       'valid_rows': len(U_GRID_VAL[(t, h)])}
                      for t in UP.TARGETS for h in UP.HORIZONS])
print(_gsz.pivot(index='target', columns='h', values='test_rows').to_string())
print(f"\n  total test evaluation points: {sum(len(v) for v in U_GRID.values()):,}")


# A SECOND registry holds every model's VALIDATION predictions. Sections U9
# (ensemble weights) and U10 (conformal calibration) fit on these, which is how
# those procedures stay strictly free of test information.
U_REG_VAL = UE.Registry(out_dir=U_OUT)


def u_register_wide(name, wide_by_h, section, notes="", grid=None, **extra):
    """Score a {(target, horizon): wide-forecast} dict on the canonical grid.

    Registers TEST predictions in `U_REG` and VALIDATION predictions in
    `U_REG_VAL` from the same forecaster.
    """
    grid = grid or U_GRID
    frames, vframes = [], []
    for _t in UP.TARGETS:
        for _h in UP.HORIZONS:
            w = wide_by_h.get((_t, _h))
            if w is None:
                continue
            frames.append(UC.to_pred_frame(w, U_CAL.wide(_t), _t, _h, grid[(_t, _h)]))
            vframes.append(UC.to_pred_frame(w, U_CAL.wide(_t), _t, _h,
                                            U_GRID_VAL[(_t, _h)]))
    U_REG_VAL.add(name, _pd.concat(vframes, ignore_index=True), section=section)
    pred = _pd.concat(frames, ignore_index=True)
    return U_REG.add(name, pred, section=section, notes=notes, **extra)


# --- persistence / seasonal naive / drift ---------------------------------
_pers, _snaive, _drift = {}, {}, {}
for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        _pers[(_t, _h)] = UC.persistence(U_CAL, _t, _h)
        _snaive[(_t, _h)] = UC.seasonal_naive(U_CAL, _t, _h)
        _drift[(_t, _h)] = UC.drift(U_CAL, _t, _h, window=28)

u_register_wide('Persistence', _pers, 'U2', 'yhat(D)=y(D-h); random walk')
u_register_wide('SeasonalNaive', _snaive, 'U2', 'yhat(D)=y(D-7); m=7')
u_register_wide('Drift', _drift, 'U2', 'random walk + trailing 28d slope')

# --- rolling mean: window selected on VALIDATION --------------------------
print("\n  Rolling-mean window selection (validation MASE, mean over h=1..3):")
_rm_sel = {}
for _t in UP.TARGETS:
    _scores = {}
    for _wnd in [3, 7, 14, 28]:
        _vals = []
        for _h in UP.HORIZONS:
            _w = UC.rolling_mean(U_CAL, _t, _h, _wnd)
            _pr = UC.to_pred_frame(_w, U_CAL.wide(_t), _t, _h,
                                   U_GRID_VAL[(_t, _h)]).dropna(subset=['yhat'])
            _vals.append(U_SCORER.score_group(_pr, _t)['MASE'])
        _scores[_wnd] = float(_np.mean(_vals))
    _best = min(_scores, key=_scores.get)
    _rm_sel[_t] = _best
    print(f"    {_t:<24} " + "  ".join(f"w={k}:{v:.4f}" for k, v in _scores.items())
          + f"   -> w={_best}")

_rmean = {}
for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        _rmean[(_t, _h)] = UC.rolling_mean(U_CAL, _t, _h, _rm_sel[_t])
u_register_wide('RollingMean', _rmean, 'U2',
                f'trailing mean, per-target window chosen on validation: {_rm_sel}',
                window_by_target=_rm_sel)

print("\n  Registered:", [k for k in U_REG.preds if U_REG.meta[k]['section'] == 'U2'])

In [ ]:
# ─── Cell U2.2 — ETS / exponential smoothing ────────────────────────────────
print("=" * 78)
print("  U2.2 — EXPONENTIAL SMOOTHING (Holt-Winters)")
print("=" * 78)
print("  Coefficients are fitted once by statsmodels on the training history;")
print("  the additive recursion is then rolled forward through validation/test,")
print("  updating on observed values only. Equivalent to a fixed-parameter")
print("  rolling forecast, but one pass instead of ~420 refits per series.\n")

_ETS_CANDS = [
    UC.ETSConfig(trend=True,  damped=False, seasonal=True),
    UC.ETSConfig(trend=True,  damped=True,  seasonal=True),
    UC.ETSConfig(trend=False, damped=False, seasonal=True),
    UC.ETSConfig(trend=True,  damped=False, seasonal=False),
]

_ets_sel, _ets_wide = {}, {}
for _t in UP.TARGETS:
    _scores = {}
    _cache = {}
    for _cfg in _ETS_CANDS:
        # SELECTION: fit on the TRAIN(fit) block only, score on VALIDATION.
        _w, _ = UC.ets_forecast_wide(U_CAL, _t, UP.HORIZONS, _cfg, U_SPLITS.train_end)
        _vals = []
        for _h in UP.HORIZONS:
            _pr = UC.to_pred_frame(_w[_h], U_CAL.wide(_t), _t, _h,
                                   U_GRID_VAL[(_t, _h)]).dropna(subset=['yhat'])
            _vals.append(U_SCORER.score_group(_pr, _t)['MASE'] if len(_pr) else _np.nan)
        _scores[_cfg.name] = float(_np.nanmean(_vals))
        _cache[_cfg.name] = _cfg
    _best_name = min(_scores, key=lambda k: _scores[k])
    _ets_sel[_t] = _best_name
    print(f"  {_t:<24}")
    for _k, _v in _scores.items():
        print(f"      {'->' if _k == _best_name else '  '} {_k:<42} valMASE={_v:.4f}")
    # FINAL: refit on the full original train period (fit + validation).
    _wf, _fits = UC.ets_forecast_wide(U_CAL, _t, UP.HORIZONS,
                                      _cache[_best_name], U_SPLITS.valid_end)
    for _h in UP.HORIZONS:
        _ets_wide[(_t, _h)] = _wf[_h]

u_register_wide('ETS', _ets_wide, 'U2',
                'Holt-Winters; per-target config selected on validation',
                config_by_target=_ets_sel)
print(f"\n  Selected ETS configs: {_ets_sel}")

In [ ]:
# ─── Cell U2.3 — ARIMA / SARIMAX ────────────────────────────────────────────
print("=" * 78)
print("  U2.3 — ARIMA (SARIMAX)")
print("=" * 78)
print("  Order is chosen per target on VALIDATION. The h-step forecast at every")
print("  origin is read exactly off the Kalman filter as  yhat(t+h|t) = Z T^h a(t|t),")
print("  verified to 1e-14 against statsmodels get_forecast(). Stationarity and")
print("  invertibility are enforced: without them the MLE can pick an explosive AR")
print("  root that looks fine at h=1 and diverges once raised to the power h.\n")

_ARIMA_CANDS = [((1, 1, 1), (0, 0, 0, 0)),
                ((2, 1, 2), (0, 0, 0, 0)),
                ((0, 1, 1), (0, 0, 0, 0)),
                ((1, 0, 0), (1, 0, 0, 7)),
                ((1, 1, 1), (1, 0, 0, 7))]

_ar_sel, _ar_wide = {}, {}
for _t in UP.TARGETS:
    _scores = {}
    for _o, _so in _ARIMA_CANDS:
        _w, _info = UC.arima_forecast_wide(U_CAL, _t, UP.HORIZONS, _o, _so,
                                           U_SPLITS.train_end)
        _errs = sum('error' in v for v in _info.values())
        _vals = []
        for _h in UP.HORIZONS:
            _pr = UC.to_pred_frame(_w[_h], U_CAL.wide(_t), _t, _h,
                                   U_GRID_VAL[(_t, _h)]).dropna(subset=['yhat'])
            _vals.append(U_SCORER.score_group(_pr, _t)['MASE'] if len(_pr) else _np.nan)
        _scores[(_o, _so)] = (float(_np.nanmean(_vals)), _errs)
    _best = min(_scores, key=lambda k: _scores[k][0] if _np.isfinite(_scores[k][0]) else 1e9)
    _ar_sel[_t] = _best
    print(f"  {_t:<24}")
    for _k, (_v, _e) in _scores.items():
        _lbl = f"ARIMA{_k[0]}x{_k[1]}"
        print(f"      {'->' if _k == _best else '  '} {_lbl:<32} "
              f"valMASE={_v:.4f}  failed_states={_e}")
    _wf, _info = UC.arima_forecast_wide(U_CAL, _t, UP.HORIZONS, _best[0], _best[1],
                                        U_SPLITS.valid_end)
    _nfail = sum('error' in v for v in _info.values())
    if _nfail:
        print(f"      NOTE: {_nfail} state(s) rejected at final fit "
              f"-> those rows are absent for ARIMA and excluded from any aligned comparison")
    for _h in UP.HORIZONS:
        _ar_wide[(_t, _h)] = _wf[_h]

u_register_wide('ARIMA', _ar_wide, 'U2',
                'SARIMAX; per-target order selected on validation',
                order_by_target={k: str(v) for k, v in _ar_sel.items()})
print(f"\n  Selected orders: " + ", ".join(f"{k}: {v[0]}x{v[1]}" for k, v in _ar_sel.items()))

In [ ]:
# ─── Cell U2.4 — Common evaluation of all statistical baselines ─────────────
print("=" * 78)
print("  U2.4 — STATISTICAL BASELINE LEADERBOARD  (test 2024-01-01 .. 2025-04-22)")
print("=" * 78)

U_BASELINE_NAMES = ['Persistence', 'SeasonalNaive', 'RollingMean', 'Drift', 'ETS', 'ARIMA']
_bl_stack = U_REG.stacked(U_BASELINE_NAMES)

print("\n  [A] Headline — observable targets only (macro-averaged across 18 states,")
print("      pooled over h=1..3). Primary ranking metric is MASE.\n")
_bl_head = U_SCORER.headline(_bl_stack)
print(_bl_head[['model', 'MASE', 'RMSSE', 'MAE', 'RMSE', 'MAPE', 'sMAPE',
                'avg_MAPE_floored_6t', 'n_obs']].to_string(index=False))

print("\n  [B] Per target x horizon MASE:\n")
_bth = U_SCORER.by_target_horizon(_bl_stack)
_piv = _bth.pivot_table(index=['target', 'horizon'], columns='model', values='MASE')
print(_piv[U_BASELINE_NAMES].round(4).to_string())

print("\n  [C] Full metric suite on the two observable targets (h pooled):\n")
_bt = U_SCORER.by_target(_bl_stack)
for _t in UP.OBSERVABLE_TARGETS:
    print(f"    --- {_t} ---")
    _s = (_bt[_bt.target == _t]
          .set_index('model')[['MAE', 'RMSE', 'MAPE', 'sMAPE', 'MASE', 'RMSSE']]
          .reindex(U_BASELINE_NAMES).round(4))
    print(_s.to_string())

_bl_head.to_csv(U_OUT / 'tables' / 'u2_baseline_headline.csv', index=False)
_bth.to_csv(U_OUT / 'tables' / 'u2_baseline_by_target_horizon.csv', index=False)
U_REG.checkpoint()          # so a later failure never costs these fits
U_REG_VAL.checkpoint('_registry_valid.pkl')

U_BEST_BASELINE = str(_bl_head.iloc[0]['model'])
U_BEST_BASELINE_MASE = float(_bl_head.iloc[0]['MASE'])
print("\n" + "=" * 78)
print(f"  STRONGEST STATISTICAL BASELINE: {U_BEST_BASELINE}  (MASE = {U_BEST_BASELINE_MASE:.4f})")
print(f"  Persistence: MASE = {float(_bl_head.set_index('model').loc['Persistence','MASE']):.4f}")
print("  This is the bar every learned model in U3-U9 must clear.")
print("=" * 78)

In [ ]:
# ─── Cell U2.5 — Baseline visualisation ─────────────────────────────────────
_fig, _axes = _plt.subplots(1, 3, figsize=(17, 4.6))

# (a) MASE by model, observable targets
_ax = _axes[0]
_d = _bt[_bt.target.isin(UP.OBSERVABLE_TARGETS)].pivot(index='model', columns='target',
                                                       values='MASE').reindex(U_BASELINE_NAMES)
_d.plot(kind='bar', ax=_ax, width=0.78, color=['#2c7fb8', '#f0a202'])
_ax.axhline(1.0, color='crimson', ls='--', lw=1.2)
_ax.text(0.02, 1.02, 'MASE = 1  (seasonal-naive skill)', transform=_ax.get_yaxis_transform(),
         color='crimson', fontsize=8, va='bottom')
_ax.set_ylabel('MASE (lower is better)'); _ax.set_xlabel('')
_ax.set_title('(a) Statistical baselines — observable targets', fontweight='bold', fontsize=10)
_ax.legend(fontsize=8); _ax.grid(axis='y', alpha=0.3)
_plt.setp(_ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)

# (b) error growth with horizon
_ax = _axes[1]
for _m in U_BASELINE_NAMES:
    _s = (_bth[(_bth.model == _m) & (_bth.target.isin(UP.OBSERVABLE_TARGETS))]
          .groupby('horizon')['MASE'].mean())
    _ax.plot(_s.index, _s.values, marker='o', label=_m, lw=1.8)
_ax.set_xlabel('forecast horizon (days ahead)'); _ax.set_ylabel('MASE')
_ax.set_xticks(UP.HORIZONS)
_ax.set_title('(b) Error growth with horizon', fontweight='bold', fontsize=10)
_ax.legend(fontsize=8); _ax.grid(alpha=0.3)

# (c) per-state persistence difficulty
_ax = _axes[2]
_ps = U_SCORER.by_state(U_REG.get('Persistence'))
_ps = _ps[_ps.target == 'total_generation_mwh'].sort_values('MASE')
_ax.barh(_ps['state_name'], _ps['MASE'], color='#4a7ba7')
_ax.set_xlabel('MASE — Persistence, total_generation_mwh')
_ax.set_title('(c) Per-state difficulty (persistence)', fontweight='bold', fontsize=10)
_ax.tick_params(axis='y', labelsize=7); _ax.grid(axis='x', alpha=0.3)

_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U2_statistical_baselines.png', dpi=150, bbox_inches='tight')
print(f"  saved -> {U_OUT / 'figures' / 'U2_statistical_baselines.png'}")
_plt.show()

---
## Section U3 — Tree / Tabular Models

Three gradient-boosting libraries on one shared, prediction-time-safe feature
set. Feature construction, training, diagnosis and evaluation all live here.

**Features** (82 per target × horizon, every one cleared by the U1.7 perturbation test)

| Block | Content |
|---|---|
| Target lags | 1, 2, 3, 7, 14, 21, 28, 30, 60, 90 — measured from the forecast origin |
| Rolling | windows 3, 7, 14, 28 × {mean, std, min, max}, ending at the origin |
| Change / volatility | Δ1, Δ2, Δ7, Δ14, acceleration, %-changes, deviation from the 7/28-day mean, 7-day z-score, 7-vs-28 trend, coefficient of variation |
| Temporal | day, weekday, week, month, quarter, year, month start/end, weekend, + cyclic sin/cos of weekday, month, day-of-year |
| Exogenous (lagged) | renewable generation, outage, consumption, coal stock, carbon intensity, renewable ratio, supply-demand gap, outage ratio — lag 1, lag 7, trailing 7-day mean |
| Cross-state | national mean at the origin, state-to-national ratio, cross-sectional rank |

**Model form.** One *global* model per (target, horizon) pooling all 18 states,
with `state_id` as a feature. Per-state models are a separate question and are
answered in Section U7.

**Protocol.** Fit on TRAIN(fit) → early-stop on VALIDATION → refit on
TRAIN + VALIDATION for the selected round count → predict TEST once.

### The scale problem, and why it decides the whole section

This panel spans three orders of magnitude — Puducherry generates ~0.7 GWh/day,
Uttar Pradesh ~368. A global model trained on raw levels minimises *pooled*
absolute error, but the headline metric is MASE **macro-averaged across states**.
Those two objectives disagree sharply here, and U3.2 measures the consequence
directly before U3.3 fixes it by learning a MASE-normalised target.

In [ ]:
# ─── Cell U3.1 — Feature build + reference fit ──────────────────────────────
from refused_upgrade import trees as UT

print("=" * 78)
print("  U3.1 — FEATURE SET")
print("=" * 78)
_f_demo = UF.build_features(U_CAL, 'total_generation_mwh', 1)
_fcols_demo = UF.feature_columns(_f_demo)
print(f"  rows (all splits) : {len(_f_demo):,}")
print(f"  admissible features: {len(_fcols_demo)}")
_blocks = {'target lags': [c for c in _fcols_demo if c.startswith('lag_')],
           'rolling': [c for c in _fcols_demo if c.startswith('roll')],
           'change/volatility': [c for c in _fcols_demo if c.startswith(('d_', 'pct_', 'dev_', 'zscore', 'trend_', 'cv', 'accel'))],
           'temporal': [c for c in _fcols_demo if c.startswith('cal_')],
           'exogenous (lagged)': [c for c in _fcols_demo if c.startswith('x_')],
           'cross-state': [c for c in _fcols_demo if c.startswith('xs_')],
           'state id': [c for c in _fcols_demo if c == 'state_id']}
for _k, _v in _blocks.items():
    print(f"    {_k:<22} {len(_v):>3}   e.g. {_v[:4]}")
del _f_demo

In [ ]:
# ─── Cell U3.2 — DIAGNOSIS: raw-level global tree vs the macro metric ───────
print("=" * 78)
print("  U3.2 — WHY A RAW-LEVEL GLOBAL TREE FAILS THE MACRO METRIC")
print("=" * 78)

_t, _h = 'total_generation_mwh', 1
_feat = UF.build_features(U_CAL, _t, _h)
_fcols = UF.feature_columns(_feat)

_fit_raw = UT.fit_predict('lgb', _feat, _fcols, seed=42)
_pr_raw = UT.pred_frame_from_fit(_feat, _fit_raw, _t, _h, 'test')

_pers_pr = U_REG.preds['Persistence']
_pers_pr = _pers_pr[(_pers_pr.target == _t) & (_pers_pr.horizon == _h)]

_a = U_SCORER.by_state(_pr_raw.assign(model='LightGBM-raw'), targets=[_t])
_b = U_SCORER.by_state(_pers_pr.assign(model='Persistence'), targets=[_t])
_cmp_state = _a.merge(_b, on=['state_name', 'target'], suffixes=('_lgb', '_pers'))
_scale_map = U_DF[U_DF['_split'] == 'train'].groupby(UP.STATE)[_t].median()
_cmp_state['train_median'] = _cmp_state['state_name'].map(_scale_map)
_cmp_state = _cmp_state.sort_values('train_median')

print(f"\n  Per-state comparison, {_t}, h=1:\n")
print(_cmp_state[['state_name', 'train_median', 'MAE_lgb', 'MAE_pers',
                  'MASE_lgb', 'MASE_pers']].round(4).to_string(index=False))

_pooled_lgb = float(_np.abs(_pr_raw.y - _pr_raw.yhat).mean())
_pooled_pers = float(_np.abs(_pers_pr.y - _pers_pr.yhat).mean())
print(f"\n  POOLED  MAE : LightGBM-raw {_pooled_lgb:.4f}   Persistence {_pooled_pers:.4f}"
      f"   -> tree WINS by {(1-_pooled_lgb/_pooled_pers)*100:.1f}%")
print(f"  MACRO   MASE: LightGBM-raw {_cmp_state.MASE_lgb.mean():.4f}   "
      f"Persistence {_cmp_state.MASE_pers.mean():.4f}   -> tree LOSES badly")
_worst = _cmp_state.loc[_cmp_state.MASE_lgb.idxmax()]
print(f"\n  The entire gap is one state: {_worst['state_name']} "
      f"(train median {_worst['train_median']:.2f} GWh/day)")
print(f"    MASE {_worst['MASE_lgb']:.2f} vs {_worst['MASE_pers']:.2f}; its absolute error is "
      f"{_worst['MAE_lgb']/_worst['MAE_pers']:.0f}x persistence's,")
print(f"    and a macro average over 18 states gives that one series 1/18 of the weight")
print(f"    regardless of its magnitude.")
print(f"\n  Extrapolation is NOT the cause here: share of test targets outside the")
print(f"  train range = {_fit_raw.extras['oob_extrapolation_rate']:.3f}.")
print("\n  FIX: learn z = y / MASE_denominator(state) instead of y. The denominator is")
print("  estimated on TRAIN only, and it makes the training loss equal the reported")
print("  metric up to a constant, so the optimiser and the leaderboard finally agree.")
del _feat

In [ ]:
# ─── Cell U3.3 — LightGBM / XGBoost / CatBoost on the MASE-normalised target ─
print("=" * 78)
print("  U3.3 — TREE MODELS (MASE-normalised global target)")
print("=" * 78)

U_TREE_KINDS = [('LightGBM', 'lgb'), ('XGBoost', 'xgb'), ('CatBoost', 'cat')]
_tree_pred = {n: [] for n, _ in U_TREE_KINDS}
_tree_val = {n: [] for n, _ in U_TREE_KINDS}
_tree_raw_pred = []
U_TREE_DIAG, U_TREE_IMP = [], {}
_t_start = _time.perf_counter()


def u_frame(rows, yhat, target, horizon):
    """Canonical prediction frame from a returned row slice."""
    return _pd.DataFrame({
        UP.DATE: rows[UP.DATE].to_numpy(), UP.STATE: rows[UP.STATE].to_numpy(),
        'target': target, 'horizon': horizon,
        'y': rows['y'].to_numpy(float), 'yhat': _np.asarray(yhat, float)})


for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        _feat = UF.build_features(U_CAL, _t, _h)
        _fcols = UF.feature_columns(_feat)
        _scale = U_SCORER.scale_for(_feat[UP.STATE].to_numpy(), _t)

        for _name, _kind in U_TREE_KINDS:
            _fit, _yh, _te, _yv, _va = UT.fit_predict_normalized(
                _kind, _feat, _fcols, _scale, seed=42)
            _tree_pred[_name].append(u_frame(_te, _yh, _t, _h))
            _tree_val[_name].append(u_frame(_va, _yv, _t, _h))
            U_TREE_DIAG.append({'model': _name, 'target': _t, 'horizon': _h,
                                'best_rounds': _fit.best_rounds})
            if _h == 1:
                U_TREE_IMP[(_name, _t)] = _fit.importance

        # keep the raw-level LightGBM as the documented counterexample
        _fit_r = UT.fit_predict('lgb', _feat, _fcols, seed=42)
        _tree_raw_pred.append(UT.pred_frame_from_fit(_feat, _fit_r, _t, _h, 'test'))
        del _feat
    print(f"    {_t:<24} done  ({_time.perf_counter()-_t_start:.0f}s elapsed)")

for _name, _ in U_TREE_KINDS:
    U_REG.add(_name, _pd.concat(_tree_pred[_name], ignore_index=True), section='U3',
              notes='global model, target normalised by the per-state MASE denominator')
    U_REG_VAL.add(_name, _pd.concat(_tree_val[_name], ignore_index=True), section='U3')
U_REG.add('LightGBM-rawlevel', _pd.concat(_tree_raw_pred, ignore_index=True), section='U3',
          notes='global model on raw levels — retained as the U3.2 counterexample')
U_TREE_NAMES = [n for n, _ in U_TREE_KINDS]
U_REG.checkpoint()
U_REG_VAL.checkpoint('_registry_valid.pkl')
print(f"\n  fitted {len(U_TREE_DIAG)+18} models in {_time.perf_counter()-_t_start:.0f}s")

In [ ]:
# ─── Cell U3.4 — Common evaluation of the tree family ──────────────────────
print("=" * 78)
print("  U3.4 — TREE MODEL EVALUATION")
print("=" * 78)

_cmp_names = U_TREE_NAMES + ['LightGBM-rawlevel', 'Persistence', 'ETS', 'ARIMA']
_tree_stack = U_REG.stacked(_cmp_names)
_tree_head = U_SCORER.headline(_tree_stack)

print("\n  [A] Headline — observable targets, macro across states, h=1..3 pooled:\n")
print(_tree_head[['model', 'MASE', 'RMSSE', 'MAE', 'RMSE', 'MAPE', 'sMAPE',
                  'avg_MAPE_floored_6t']].to_string(index=False))

_pers_mase = float(_tree_head.set_index('model').loc['Persistence', 'MASE'])
print(f"\n  vs Persistence (MASE {_pers_mase:.4f}):")
for _, _r in _tree_head.iterrows():
    if _r['model'] in U_TREE_NAMES + ['LightGBM-rawlevel']:
        _d = (_r['MASE'] - _pers_mase) / _pers_mase * 100
        print(f"    {_r['model']:<20} {_r['MASE']:.4f}  ({_d:+6.2f}%)  "
              f"{'BEATS' if _r['MASE'] < _pers_mase else 'does NOT beat'} persistence")

print("\n  [B] MASE by target x horizon:\n")
_tth = U_SCORER.by_target_horizon(_tree_stack)
print(_tth.pivot_table(index=['target', 'horizon'], columns='model',
                       values='MASE')[_cmp_names].round(4).to_string())

print("\n  [C] Boosting rounds selected on validation (mean over horizons):\n")
_diag = _pd.DataFrame(U_TREE_DIAG)
print(_diag.pivot_table(index='target', columns='model',
                        values='best_rounds', aggfunc='mean').round(0).to_string())

_tree_head.to_csv(U_OUT / 'tables' / 'u3_tree_headline.csv', index=False)
_tth.to_csv(U_OUT / 'tables' / 'u3_tree_by_target_horizon.csv', index=False)
_diag.to_csv(U_OUT / 'tables' / 'u3_tree_diagnostics.csv', index=False)
_cmp_state.to_csv(U_OUT / 'tables' / 'u3_scale_pathology_by_state.csv', index=False)

In [ ]:
# ─── Cell U3.5 — Feature importance and figure ─────────────────────────────
print("=" * 78)
print("  U3.5 — WHAT THE TREES USE  (LightGBM gain, h=1)")
print("=" * 78)
for _t in UP.OBSERVABLE_TARGETS:
    _imp = U_TREE_IMP[('LightGBM', _t)]
    _top = _imp.head(10) / _imp.sum() * 100
    print(f"\n  --- {_t} ---")
    for _f, _v in _top.items():
        print(f"      {_f:<26} {_v:5.1f}%  {'#' * max(1, int(_v))}")

_fig, _axes = _plt.subplots(1, 3, figsize=(17, 4.6))

_ax = _axes[0]
_d = _tree_head.set_index('model').reindex(_cmp_names)['MASE']
_cols = ['#2c7fb8' if m in U_TREE_NAMES else ('#c0392b' if m == 'LightGBM-rawlevel' else '#8a8a8a')
         for m in _d.index]
_ax.bar(range(len(_d)), _d.values, color=_cols)
_ax.axhline(_pers_mase, color='crimson', ls='--', lw=1.3)
_ax.set_xticks(range(len(_d)))
_ax.set_xticklabels(_d.index, rotation=35, ha='right', fontsize=7.5)
_ax.set_ylabel('MASE (observable targets)')
_ax.set_title('(a) Trees vs baselines', fontweight='bold', fontsize=10)
_ax.grid(axis='y', alpha=0.3)

_ax = _axes[1]
_ax.scatter(_cmp_state['train_median'], _cmp_state['MASE_lgb'],
            s=55, color='#c0392b', label='LightGBM (raw level)')
_ax.scatter(_cmp_state['train_median'], _cmp_state['MASE_pers'],
            s=55, color='#2c7fb8', marker='s', label='Persistence')
_ax.set_xscale('log'); _ax.set_yscale('log')
_ax.set_xlabel('state train median generation (GWh/day, log)')
_ax.set_ylabel('MASE (log)')
_ax.set_title('(b) The scale pathology', fontweight='bold', fontsize=10)
_ax.legend(fontsize=8); _ax.grid(alpha=0.3, which='both')

_ax = _axes[2]
for _m in U_TREE_NAMES + ['Persistence']:
    _s = (_tth[(_tth.model == _m) & (_tth.target.isin(UP.OBSERVABLE_TARGETS))]
          .groupby('horizon')['MASE'].mean())
    _ax.plot(_s.index, _s.values, marker='o', lw=1.8, label=_m)
_ax.set_xlabel('horizon (days)'); _ax.set_ylabel('MASE'); _ax.set_xticks(UP.HORIZONS)
_ax.set_title('(c) Error growth with horizon', fontweight='bold', fontsize=10)
_ax.legend(fontsize=8); _ax.grid(alpha=0.3)

_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U3_tree_models.png', dpi=150, bbox_inches='tight')
print(f"\n  saved -> {U_OUT / 'figures' / 'U3_tree_models.png'}")
_plt.show()

---
## Section U4 — Hybrid Residual Forecasting

Section U2 showed persistence is the strongest baseline; Section U3 showed a
tree predicting the *level* cannot survive the train→test level shift. The
hybrid resolves both at once:

```
  base   = a statistical forecast that handles the level  (persistence, snaive, ETS)
  r      = y − base                                        (the part the base misses)
  r̂      = gradient-boosted model on the safe feature set
  ŷ      = base + r̂
```

Two properties make this the right structure for this dataset:

1. **The level problem disappears.** `r` is a *change*, which is close to
   stationary, so the tree is never asked to extrapolate beyond its training
   range even when the level doubles.
2. **The baseline is a floor, not a rival.** If the model learns nothing,
   `r̂ ≈ 0` and the hybrid degenerates to its base. It can only lose by
   predicting badly, and validation-based early stopping guards that.

**Variants** — A: Persistence + LightGBM · B: Seasonal-naive + LightGBM ·
C: ETS + LightGBM · D: best U2 baseline + XGBoost.

**Base-model honesty.** ETS/ARIMA bases used here are fitted on the TRAIN(fit)
block *only*, so the residuals the model learns from on TRAIN come from the
same parameterisation that generates the TEST base. Their recursive state still
updates on observed data through the test period; only the smoothing
coefficients are frozen. Persistence and seasonal-naive have no fitted
parameters at all, so variants A and B are entirely free of this concern.

In [ ]:
# ─── Cell U4.1 — Build the base forecasts ───────────────────────────────────
from refused_upgrade import trees as UT

print("=" * 78)
print("  U4.1 — BASE FORECASTS FOR THE HYBRIDS")
print("=" * 78)

U_BASE_WIDE = {}

# Parameter-free bases: defined on every date, no fitting, no leakage surface.
for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        U_BASE_WIDE[('Persistence', _t, _h)] = UC.persistence(U_CAL, _t, _h)
        U_BASE_WIDE[('SeasonalNaive', _t, _h)] = UC.seasonal_naive(U_CAL, _t, _h)
print("  Persistence / SeasonalNaive bases ready (parameter-free).")

# ETS base refitted on TRAIN(fit) only, using the configuration U2.2 selected.
_ETS_CFG_LOOKUP = {
    'ETS(trend=add,seasonal=add)':    UC.ETSConfig(True, False, True),
    'ETS(trend=damped,seasonal=add)': UC.ETSConfig(True, True, True),
    'ETS(trend=none,seasonal=add)':   UC.ETSConfig(False, False, True),
    'ETS(trend=add,seasonal=none)':   UC.ETSConfig(True, False, False),
}
_t0 = _time.perf_counter()
for _t in UP.TARGETS:
    _cfg = _ETS_CFG_LOOKUP[U_REG.meta['ETS']['config_by_target'][_t]]
    _w, _ = UC.ets_forecast_wide(U_CAL, _t, UP.HORIZONS, _cfg, U_SPLITS.train_end)
    for _h in UP.HORIZONS:
        U_BASE_WIDE[('ETS', _t, _h)] = _w[_h]
print(f"  ETS base refitted on TRAIN(fit) only  ({_time.perf_counter()-_t0:.0f}s)")

print(f"\n  Base coverage on the test grid (share of grid rows with a usable base):")
for _b in ['Persistence', 'SeasonalNaive', 'ETS']:
    _cov = []
    for _t in UP.TARGETS:
        for _h in UP.HORIZONS:
            _g = U_GRID[(_t, _h)]
            _bw = U_BASE_WIDE[(_b, _t, _h)]
            _s = _bw.stack(future_stack=True).rename('b').reset_index()
            _s.columns = [UP.DATE, UP.STATE, 'b']
            _cov.append(_g.merge(_s, on=[UP.DATE, UP.STATE], how='left')['b'].notna().mean())
    print(f"    {_b:<16} {_np.mean(_cov)*100:6.2f}%")

In [ ]:
# ─── Cell U4.2 — Fit the four hybrid variants ──────────────────────────────
print("=" * 78)
print("  U4.2 — HYBRID RESIDUAL MODELS")
print("=" * 78)

U_HYBRIDS = [
    ('Hybrid_Persistence_LGBM',  'Persistence',   'lgb'),
    ('Hybrid_SeasonalNaive_LGBM', 'SeasonalNaive', 'lgb'),
    ('Hybrid_ETS_LGBM',          'ETS',           'lgb'),
    ('Hybrid_Persistence_XGB',   'Persistence',   'xgb'),   # variant D: best U2 base
]


def u_attach_base(feat, base_wide):
    """Add the base forecast as a column, aligned on (date, state)."""
    _s = base_wide.stack(future_stack=True).rename('base').reset_index()
    _s.columns = [UP.DATE, UP.STATE, 'base']
    out = feat.merge(_s, on=[UP.DATE, UP.STATE], how='left')
    return out[out['base'].notna()].reset_index(drop=True)


_hyb_pred = {n: [] for n, _, _ in U_HYBRIDS}
_hyb_val = {n: [] for n, _, _ in U_HYBRIDS}
_hyb_diag = []
_t_start = _time.perf_counter()

for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        _feat0 = UF.build_features(U_CAL, _t, _h)
        for _name, _basename, _kind in U_HYBRIDS:
            _feat = u_attach_base(_feat0, U_BASE_WIDE[(_basename, _t, _h)])
            # `base` and its gap to the last observation are both known at the
            # forecast origin, so both are admissible inputs to the residual model.
            _feat['base_minus_lag1'] = _feat['base'] - _feat['lag_1']
            _fcols = UF.feature_columns(_feat)
            _scale = U_SCORER.scale_for(_feat[UP.STATE].to_numpy(), _t)

            # The model learns  z = (y - base) / MASE_denominator(state).
            # Removing the base kills the level shift; dividing by the MASE
            # denominator kills the cross-state scale spread that U3.2 showed
            # is fatal to the macro metric. The residual needs BOTH: it inherits
            # the scale pathology just as the raw level does.
            _fit, _yhat, _te, _yval, _va = UT.fit_predict_normalized(
                _kind, _feat, _fcols, _scale,
                base=_feat['base'].to_numpy(float), seed=42)

            # --- validation-calibrated shrinkage -----------------------------
            # A residual model can only help where the base leaves exploitable
            # structure. On avg_market_price the base is already near-perfect
            # (persistence MASE ~0.08), so an uncalibrated correction injects
            # more noise than it removes. lambda* is chosen on VALIDATION only:
            #   yhat = base + lambda * residual_hat,  lambda in [0, 1]
            # lambda*=0 returns the pure base, so the hybrid is protected from
            # ever being much worse than the baseline it is built on, and the
            # test split plays no part in the choice.
            _va_base = _va['base'].to_numpy(float)
            _va_resid_hat = _yval - _va_base
            _best_lam, _best_sc = 0.0, _np.inf
            for _lam in _np.linspace(0, 1, 11):
                _cand = _pd.DataFrame({
                    UP.DATE: _va[UP.DATE].to_numpy(), UP.STATE: _va[UP.STATE].to_numpy(),
                    'target': _t, 'horizon': _h, 'y': _va['y'].to_numpy(float),
                    'yhat': _va_base + _lam * _va_resid_hat})
                _s = U_SCORER.score_group(_cand, _t)['MASE']
                if _s < _best_sc:
                    _best_lam, _best_sc = float(_lam), float(_s)

            _yhat = _te['base'].to_numpy(float) + _best_lam * (
                _yhat - _te['base'].to_numpy(float))
            _yval_shrunk = _va_base + _best_lam * _va_resid_hat
            _hyb_pred[_name].append(_pd.DataFrame({
                UP.DATE: _te[UP.DATE].to_numpy(), UP.STATE: _te[UP.STATE].to_numpy(),
                'target': _t, 'horizon': _h,
                'y': _te['y'].to_numpy(float), 'yhat': _yhat}))
            _hyb_val[_name].append(_pd.DataFrame({
                UP.DATE: _va[UP.DATE].to_numpy(), UP.STATE: _va[UP.STATE].to_numpy(),
                'target': _t, 'horizon': _h,
                'y': _va['y'].to_numpy(float), 'yhat': _yval_shrunk}))

            _r_te = (_te['y'] - _te['base']).to_numpy(float)
            _pred_resid = _yhat - _te['base'].to_numpy(float)
            _hyb_diag.append({
                'model': _name, 'target': _t, 'horizon': _h,
                'best_rounds': _fit.best_rounds,
                'lambda_valid': _best_lam,
                'valid_MASE_at_lambda': round(_best_sc, 4),
                'resid_std_test': float(_np.std(_r_te)),
                'corr_pred_resid': float(_np.corrcoef(_pred_resid, _r_te)[0, 1])
                if (len(_r_te) > 2 and _np.std(_pred_resid) > 0) else _np.nan,
            })
        del _feat0
    print(f"    {_t:<24} done  ({_time.perf_counter()-_t_start:.0f}s elapsed)")

for _name, _, _ in U_HYBRIDS:
    U_REG.add(_name, _pd.concat(_hyb_pred[_name], ignore_index=True),
              section='U4', notes='base + gradient-boosted residual, '
                                  'validation-calibrated shrinkage')
    U_REG_VAL.add(_name, _pd.concat(_hyb_val[_name], ignore_index=True), section='U4')
U_HYBRID_NAMES = [n for n, _, _ in U_HYBRIDS]
U_REG.checkpoint()
U_REG_VAL.checkpoint('_registry_valid.pkl')
print(f"\n  fitted {len(_hyb_diag)} hybrid models in {_time.perf_counter()-_t_start:.0f}s")

In [ ]:
# ─── Cell U4.3 — Common evaluation of the hybrid family ────────────────────
print("=" * 78)
print("  U4.3 — HYBRID EVALUATION")
print("=" * 78)

_cmp = U_HYBRID_NAMES + ['Persistence', 'SeasonalNaive', 'ETS', 'ARIMA', 'LightGBM']
_hy_stack = U_REG.stacked(_cmp)
_hy_head = U_SCORER.headline(_hy_stack)

print("\n  [A] Headline — observable targets, macro across states, h=1..3 pooled:\n")
print(_hy_head[['model', 'MASE', 'RMSSE', 'MAE', 'RMSE', 'MAPE', 'sMAPE',
                'avg_MAPE_floored_6t', 'n_obs']].to_string(index=False))

_pers_mase = float(_hy_head.set_index('model').loc['Persistence', 'MASE'])
print(f"\n  Persistence reference MASE = {_pers_mase:.4f}")
print("  Improvement over persistence (negative = better):\n")
for _, _r in _hy_head.iterrows():
    if _r['model'] in U_HYBRID_NAMES:
        _d = (_r['MASE'] - _pers_mase) / _pers_mase * 100
        _v = 'BEATS persistence' if _r['MASE'] < _pers_mase else 'does NOT beat persistence'
        print(f"    {_r['model']:<28} MASE={_r['MASE']:.4f}  ({_d:+.2f}%)  {_v}")

print("\n  [B] MASE by target x horizon:\n")
_hth = U_SCORER.by_target_horizon(_hy_stack)
print(_hth.pivot_table(index=['target', 'horizon'], columns='model',
                       values='MASE')[_cmp].round(4).to_string())

print("\n  [C] Validation-calibrated shrinkage lambda* per (target, horizon).")
print("      lambda*=0 means the residual model was rejected on validation and the")
print("      hybrid fell back to its pure base; lambda*=1 means it was accepted whole.\n")
_hd = _pd.DataFrame(_hyb_diag)
print(_hd.pivot_table(index=['target', 'horizon'], columns='model',
                      values='lambda_valid').round(2).to_string())

print("\n  [D] Does the residual model carry real signal?")
print("      Correlation between the predicted and realised test residual")
print("      (~0 means nothing was learned beyond the base):\n")
print(_hd.pivot_table(index='horizon', columns='model',
                      values='corr_pred_resid').round(3).to_string())

_hy_head.to_csv(U_OUT / 'tables' / 'u4_hybrid_headline.csv', index=False)
_hth.to_csv(U_OUT / 'tables' / 'u4_hybrid_by_target_horizon.csv', index=False)
_hd.to_csv(U_OUT / 'tables' / 'u4_hybrid_diagnostics.csv', index=False)

U_BEST_HYBRID = str(_hy_head[_hy_head.model.isin(U_HYBRID_NAMES)].iloc[0]['model'])
print(f"\n  BEST HYBRID: {U_BEST_HYBRID}")

In [ ]:
# ─── Cell U4.4 — Hybrid visualisation ───────────────────────────────────────
_fig, _axes = _plt.subplots(1, 3, figsize=(17, 4.6))

_ax = _axes[0]
_d = _hy_head.set_index('model').reindex(_cmp)['MASE']
_cols = ['#1a7f37' if m in U_HYBRID_NAMES else '#8a8a8a' for m in _d.index]
_ax.bar(range(len(_d)), _d.values, color=_cols)
_ax.axhline(_pers_mase, color='crimson', ls='--', lw=1.4)
_ax.text(len(_d) - 0.4, _pers_mase, ' persistence', color='crimson', fontsize=8, va='bottom', ha='right')
_ax.set_xticks(range(len(_d)))
_ax.set_xticklabels([m.replace('Hybrid_', 'H:') for m in _d.index], rotation=35, ha='right', fontsize=7.5)
_ax.set_ylabel('MASE'); _ax.set_title('(a) Hybrids vs their bases', fontweight='bold', fontsize=10)
_ax.grid(axis='y', alpha=0.3)

_ax = _axes[1]
for _m in [U_BEST_HYBRID, 'Persistence', 'ETS', 'LightGBM']:
    _s = (_hth[(_hth.model == _m) & (_hth.target.isin(UP.OBSERVABLE_TARGETS))]
          .groupby('horizon')['MASE'].mean())
    _ax.plot(_s.index, _s.values, marker='o', lw=2, label=_m.replace('Hybrid_', 'H:'))
_ax.set_xlabel('horizon (days)'); _ax.set_ylabel('MASE'); _ax.set_xticks(UP.HORIZONS)
_ax.set_title('(b) Gain concentrates at longer horizons', fontweight='bold', fontsize=10)
_ax.legend(fontsize=8); _ax.grid(alpha=0.3)

_ax = _axes[2]
_bs = U_SCORER.by_state(U_REG.get(U_BEST_HYBRID, 'Persistence'))
_bs = _bs[_bs.target == 'total_generation_mwh'].pivot(index='state_name',
                                                      columns='model', values='MASE')
_bs['delta_%'] = (_bs[U_BEST_HYBRID] - _bs['Persistence']) / _bs['Persistence'] * 100
_bs = _bs.sort_values('delta_%')
_ax.barh(_bs.index, _bs['delta_%'],
         color=['#1a7f37' if v < 0 else '#c0392b' for v in _bs['delta_%']])
_ax.axvline(0, color='black', lw=1)
_ax.set_xlabel('MASE change vs persistence (%)  — negative is better')
_ax.set_title('(c) Per-state effect, generation', fontweight='bold', fontsize=10)
_ax.tick_params(axis='y', labelsize=7); _ax.grid(axis='x', alpha=0.3)

_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U4_hybrid_residual.png', dpi=150, bbox_inches='tight')
print(f"  saved -> {U_OUT / 'figures' / 'U4_hybrid_residual.png'}")
_plt.show()

---
## Section U5 — New Deep Time-Series Models

Three modern architectures, none of which exist in the original notebook:

| Model | Idea |
|---|---|
| **N-BEATS** | residual stack of fully-connected blocks; each block subtracts what it can explain and adds its forecast |
| **N-HiTS** | N-BEATS with multi-rate input pooling, so different stacks specialise on different frequencies |
| **TSMixer** | all-MLP: alternating time-mixing and channel-mixing layers with residual connections |

The existing `PatchTST` / `MCAG` models are **not** touched, re-run or replaced;
they are reproduced separately in Section U6 for a like-for-like comparison.

### Two design decisions, both forced by Section U3.2

**Instance normalisation.** Every window is normalised as
`x ← (x − last_observed_value) / window_std` and the forecast is mapped back
the same way. This (a) removes the train→test level shift, and (b) puts
Puducherry (~0.7 GWh/day) and Uttar Pradesh (~368) on one scale, without which
a global model is fitted almost entirely to the largest states. Because the
offset is the *last observed value*, **a model that outputs zero is exactly the
persistence forecast** — so these models start from the same floor as the
Section U4 hybrids and any gain is genuinely learned.

**Fully-observed windows only.** A 28-day lookback is required to be complete;
no value is ever fabricated to fill a window. The test period contains one
54-day outage (2024-12-11 → 2025-02-02), so this costs coverage immediately
after that block. Coverage is reported below, and the final leaderboard scores
every model on the intersection of all models' evaluation points.

In [ ]:
# ─── Cell U5.1 — Windowed panel datasets ────────────────────────────────────
from refused_upgrade import deep as UD

print("=" * 78)
print("  U5.1 — WINDOWED PANEL DATASETS")
print("=" * 78)
print(f"  device: {UD.DEVICE}")

U_SPEC = UD.WindowSpec(lookback=28, horizons=tuple(UP.HORIZONS), calendar=True)
_ds_tr, _ds_va, _ds_te = UD.build_datasets(U_CAL, 'total_generation_mwh', U_SPEC)
print(f"  lookback={U_SPEC.lookback}  horizons={U_SPEC.horizons}")
print(f"  windows: train={len(_ds_tr):,}  valid={len(_ds_va):,}  test={len(_ds_te):,}")
_X0, _y0, _s0, _n0, _t0i = _ds_tr[0]
U_NFEAT = _X0.shape[1]
print(f"  input tensor per window: {tuple(_X0.shape)}  "
      f"(1 target channel + 6 calendar channels)")
print(f"  test grid rows at h=1: {len(U_GRID[('total_generation_mwh', 1)]):,}  "
      f"-> window coverage ~{len(_ds_te)/max(len(U_GRID[('total_generation_mwh',1)]),1)*100:.0f}%")

In [ ]:
# ─── Cell U5.2 — Train N-BEATS / N-HiTS / TSMixer ──────────────────────────
print("=" * 78)
print("  U5.2 — TRAINING DEEP MODELS")
print("=" * 78)

U_DEEP_BUILDERS = {
    'N-BEATS': lambda: UD.NBeats(U_SPEC.lookback, len(U_SPEC.horizons),
                                 n_blocks=4, width=256, n_feat=U_NFEAT),
    'N-HiTS':  lambda: UD.NHiTS(U_SPEC.lookback, len(U_SPEC.horizons),
                                pool_sizes=(8, 4, 1), width=256, n_feat=U_NFEAT),
    'TSMixer': lambda: UD.TSMixer(U_SPEC.lookback, len(U_SPEC.horizons),
                                  n_feat=U_NFEAT, n_blocks=4, hidden=128),
}

U_DEEP_CFG = UD.TrainConfig(epochs=40, batch_size=512, lr=1e-3,
                            patience=8, loss='l1', seed=42, verbose=False)

_deep_pred = {k: [] for k in U_DEEP_BUILDERS}
_deep_val = {k: [] for k in U_DEEP_BUILDERS}
U_DEEP_HIST = {}
_t_start = _time.perf_counter()

for _t in UP.TARGETS:
    _dtr, _dva, _dte = UD.build_datasets(U_CAL, _t, U_SPEC)
    for _name, _build in U_DEEP_BUILDERS.items():
        UD.set_seed(42)
        _model = _build()
        _model, _hist = UD.train_model(_model, _dtr, _dva, U_DEEP_CFG)
        _pr = UD.predict(_model, _dte, U_SPEC, U_CAL.dates, U_CAL.states, _t)
        # restrict to the canonical evaluation grid
        _keep = []
        for _h in UP.HORIZONS:
            _g = U_GRID[(_t, _h)][[UP.DATE, UP.STATE]]
            _keep.append(_pr[_pr.horizon == _h].merge(_g, on=[UP.DATE, UP.STATE], how='inner'))
        _deep_pred[_name].append(_pd.concat(_keep, ignore_index=True))
        # validation predictions feed the U9 ensemble weights and U10 conformal
        # calibration; the model never saw this block during training
        _pv = UD.predict(_model, _dva, U_SPEC, U_CAL.dates, U_CAL.states, _t)
        _keepv = [_pv[_pv.horizon == _h].merge(U_GRID_VAL[(_t, _h)][[UP.DATE, UP.STATE]],
                                               on=[UP.DATE, UP.STATE], how='inner')
                  for _h in UP.HORIZONS]
        _deep_val[_name].append(_pd.concat(_keepv, ignore_index=True))
        U_DEEP_HIST[(_name, _t)] = _hist
        del _model
    print(f"    {_t:<24} done  ({_time.perf_counter()-_t_start:.0f}s elapsed)")

for _name in U_DEEP_BUILDERS:
    U_REG.add(_name, _pd.concat(_deep_pred[_name], ignore_index=True), section='U5',
              notes=f'global model, instance-normalised windows, lookback={U_SPEC.lookback}')
    U_REG_VAL.add(_name, _pd.concat(_deep_val[_name], ignore_index=True), section='U5')
U_DEEP_NAMES = list(U_DEEP_BUILDERS)
U_REG.checkpoint()
U_REG_VAL.checkpoint('_registry_valid.pkl')
print(f"\n  trained {len(UP.TARGETS)*len(U_DEEP_BUILDERS)} deep models "
      f"in {_time.perf_counter()-_t_start:.0f}s")

In [ ]:
# ─── Cell U5.3 — Common evaluation of the deep family ──────────────────────
print("=" * 78)
print("  U5.3 — DEEP MODEL EVALUATION")
print("=" * 78)

_cmp = U_DEEP_NAMES + ['Persistence', 'ETS', 'LightGBM'] + \
       [n for n in ['Hybrid_Persistence_LGBM'] if n in U_REG.preds]
_deep_stack = U_REG.stacked(_cmp)
_deep_head = U_SCORER.headline(_deep_stack)

print("\n  NOTE: models are aligned onto their common evaluation points before")
print("  scoring, so the baseline numbers here are computed on the deep models'")
print(f"  window-restricted grid ({int(_deep_head['n_obs'].iloc[0]):,} rows) rather than the")
print("  full grid used in U2-U4. Compare rows within this table, not across tables.\n")

print(_deep_head[['model', 'MASE', 'RMSSE', 'MAE', 'RMSE', 'MAPE', 'sMAPE',
                  'avg_MAPE_floored_6t', 'n_obs']].to_string(index=False))

_pers_m = float(_deep_head.set_index('model').loc['Persistence', 'MASE'])
print(f"\n  vs Persistence (MASE {_pers_m:.4f}) on this aligned grid:")
for _, _r in _deep_head.iterrows():
    if _r['model'] in U_DEEP_NAMES:
        _d = (_r['MASE'] - _pers_m) / _pers_m * 100
        print(f"    {_r['model']:<12} {_r['MASE']:.4f}  ({_d:+6.2f}%)  "
              f"{'BEATS' if _r['MASE'] < _pers_m else 'does NOT beat'} persistence")

print("\n  MASE by target x horizon:\n")
_dth = U_SCORER.by_target_horizon(_deep_stack)
print(_dth.pivot_table(index=['target', 'horizon'], columns='model',
                       values='MASE')[_cmp].round(4).to_string())

_deep_head.to_csv(U_OUT / 'tables' / 'u5_deep_headline.csv', index=False)
_dth.to_csv(U_OUT / 'tables' / 'u5_deep_by_target_horizon.csv', index=False)

In [ ]:
# ─── Cell U5.4 — Deep model visualisation ───────────────────────────────────
_fig, _axes = _plt.subplots(1, 3, figsize=(17, 4.6))

_ax = _axes[0]
_d = _deep_head.set_index('model').reindex(_cmp)['MASE']
_cols = ['#6a3d9a' if m in U_DEEP_NAMES else '#8a8a8a' for m in _d.index]
_ax.bar(range(len(_d)), _d.values, color=_cols)
_ax.axhline(_pers_m, color='crimson', ls='--', lw=1.3)
_ax.set_xticks(range(len(_d)))
_ax.set_xticklabels([m.replace('Hybrid_', 'H:') for m in _d.index],
                    rotation=35, ha='right', fontsize=7.5)
_ax.set_ylabel('MASE'); _ax.set_title('(a) Deep models vs references', fontweight='bold', fontsize=10)
_ax.grid(axis='y', alpha=0.3)

_ax = _axes[1]
for _name in U_DEEP_NAMES:
    _h = U_DEEP_HIST[(_name, 'total_generation_mwh')]
    _ax.plot(_h['valid'], lw=1.8, label=f"{_name} (valid)")
_ax.set_xlabel('epoch'); _ax.set_ylabel('validation L1 (normalised units)')
_ax.set_title('(b) Convergence — generation', fontweight='bold', fontsize=10)
_ax.legend(fontsize=8); _ax.grid(alpha=0.3)

_ax = _axes[2]
for _m in U_DEEP_NAMES + ['Persistence']:
    _s = (_dth[(_dth.model == _m) & (_dth.target.isin(UP.OBSERVABLE_TARGETS))]
          .groupby('horizon')['MASE'].mean())
    _ax.plot(_s.index, _s.values, marker='o', lw=1.8, label=_m)
_ax.set_xlabel('horizon (days)'); _ax.set_ylabel('MASE'); _ax.set_xticks(UP.HORIZONS)
_ax.set_title('(c) Error growth with horizon', fontweight='bold', fontsize=10)
_ax.legend(fontsize=8); _ax.grid(alpha=0.3)

_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U5_deep_models.png', dpi=150, bbox_inches='tight')
print(f"  saved -> {U_OUT / 'figures' / 'U5_deep_models.png'}")
_plt.show()

---
## Section U6 — MCAG-v2 / RE-FUSED-Specific Improvements

**The existing MCAG is not modified.** `refused_upgrade.mcag` contains
*re-implementations* of `MCAG` and `MCAG_Regime` written from the Cell 3.2
definitions, so new variants can be built without importing, patching or
re-running a single original cell. The published `PatchTST+MCAG` numbers stand
exactly as they are.

This section asks two separate questions.

**U6.1 — Does the original architecture, scored on this grid, behave as the
published numbers suggest?** A replica is trained on the *original* 25-feature
input with the *original* global standardisation. If the U1.5 diagnosis is
right, it should land far above persistence — confirming the problem was the
input specification, not the transformer.

**U6.2–U6.4 — Does carbon-aware gating add anything once the specification is
fixed?** The same gating idea is rebuilt on instance-normalised windows (so a
zero output is the persistence forecast) and extended six ways:

| | Extension |
|---|---|
| A | state embedding |
| B | regime embedding (the six train-only regime flags) |
| C | cross-state lagged information (national mean at the origin) |
| D | improved carbon/price fusion (separate gates + learned mixing) |
| E | multi-task prediction (auxiliary heads on the other targets) |
| F | best practical combination, chosen on **validation** |

In [ ]:
# ─── Cell U6.1 — Reference: the original architecture on this grid ──────────
import torch as _torch
from refused_upgrade import mcag as UM
from refused_upgrade import deep as UD

print("=" * 78)
print("  U6.1 — ORIGINAL PatchTST+MCAG, REPRODUCED ON THE UPGRADE GRID")
print("=" * 78)
print("  Input: the original Cell 3.1 FEATS (no target lag), globally standardised")
print("  with train-only mean/std — i.e. the original specification, windowed")
print("  calendar-correctly per state so it can be scored against everything else.\n")

_ORIG_FEATS = ['month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'renewable_ratio',
               'outage_ratio', 'carbon_intensity', 'carbon_budget_pressure',
               'grid_stress_index', 'gen_efficiency', 'price_norm',
               're_penetration_pct', 'demand_pressure', 'coal_outage_stress',
               'p_carbon_gcal', 'lcmp_carbon', 'discom_stress', 'freq_excursion_risk',
               'monsoon_re_risk', 'ppa_deviation', 'rolling_cvar4hr',
               'ci_intraday_variance', 're_ramp_risk', 'monsoon_re_credit', 'pa_lmp_t']
_ORIG_FEATS = [c for c in _ORIG_FEATS if c in U_DF.columns]
_ORIG_CARBON = [c for c in ['p_carbon_gcal', 'lcmp_carbon', 'carbon_budget_pressure',
                            'freq_excursion_risk', 'monsoon_re_risk'] if c in _ORIG_FEATS]
print(f"  original features: {len(_ORIG_FEATS)}   carbon gate inputs: {len(_ORIG_CARBON)}")
print(f"  contains a lag of total_generation_mwh: "
      f"{any('gen' in f and 'lag' in f for f in _ORIG_FEATS)}")


class OrigFeatWindows(UD.Dataset):
    """Windows over the ORIGINAL feature specification.

    Standardisation is global (train mean/std per column), exactly as Cell 3.1
    did — deliberately NOT the instance normalisation used elsewhere in Part II,
    because the point of this cell is to reproduce the original setup.
    """

    def __init__(self, cal, feats, carbon, target, lookback, horizons, split, stats):
        self.L, self.H = lookback, max(horizons)
        self.horizons = tuple(horizons)
        self.dates, self.states = cal.dates, cal.states
        mu, sd = stats
        self.X = _np.stack([((cal.wide(f).to_numpy(float) - mu[f]) / sd[f])
                            for f in feats], axis=-1)          # (T, S, F)
        self.cidx = [feats.index(c) for c in carbon]
        self.Y = cal.wide(target).to_numpy(float)
        self.ymu, self.ysd = float(mu[target]), float(sd[target])
        S = cal.split_of().to_numpy(dtype=object)
        obs = _np.isfinite(self.Y) & _np.isfinite(self.X).all(axis=-1)
        cobs = _np.cumsum(obs.astype(int), axis=0)
        idx = []
        for s in range(self.Y.shape[1]):
            for t in range(self.L - 1, self.Y.shape[0] - self.H):
                if (cobs[t, s] - (cobs[t - self.L, s] if t - self.L >= 0 else 0)) != self.L:
                    continue
                if not _np.isfinite(self.Y[t + 1:t + 1 + self.H, s]).all():
                    continue
                if S[t + self.H, s] != split:
                    continue
                idx.append((s, t))
        self.index = idx

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        import torch
        s, t = self.index[i]
        sl = slice(t - self.L + 1, t + 1)
        x = _np.nan_to_num(self.X[sl, s, :]).astype('f4')
        c = x[:, self.cidx]
        y = ((self.Y[t + 1:t + 1 + self.H, s] - self.ymu) / self.ysd).astype('f4')
        return (torch.from_numpy(x), torch.from_numpy(c), torch.from_numpy(y),
                torch.tensor(s), torch.tensor(t))


_tr_rows = U_DF[U_DF['_split'] == 'train']
_STAT_COLS = list(dict.fromkeys(_ORIG_FEATS + UP.TARGETS))   # dedupe: GSI is both
_ORIG_STATS = (_tr_rows[_STAT_COLS].mean(),
               _tr_rows[_STAT_COLS].std().replace(0, 1))

_orig_pred = []
_t0 = _time.perf_counter()
for _t in UP.TARGETS:
    _dsets = [OrigFeatWindows(U_CAL, _ORIG_FEATS, _ORIG_CARBON, _t, 28,
                              UP.HORIZONS, sp, _ORIG_STATS)
              for sp in ('train', 'valid', 'test')]
    UD.set_seed(42)
    _m = UM.PatchTSTMCAG_Replica(28, len(_ORIG_FEATS), len(_ORIG_CARBON),
                                 len(UP.HORIZONS), n_targets=1, d=32).to(UD.DEVICE)
    _opt = _torch.optim.AdamW(_m.parameters(), lr=1e-3, weight_decay=1e-5)
    _dl_tr = UD.DataLoader(_dsets[0], batch_size=512, shuffle=True, drop_last=True)
    _dl_va = UD.DataLoader(_dsets[1], batch_size=512)
    _best, _bstate, _bad = 1e9, None, 0
    for _ep in range(40):
        _m.train()
        for _x, _c, _y, _s, _ti in _dl_tr:
            _opt.zero_grad()
            _loss = _torch.nn.functional.l1_loss(
                _m(_x.to(UD.DEVICE), _c.to(UD.DEVICE)).squeeze(-1), _y.to(UD.DEVICE))
            _loss.backward(); _opt.step()
        _m.eval(); _tot = _n = 0
        with _torch.no_grad():
            for _x, _c, _y, _s, _ti in _dl_va:
                _l = _torch.nn.functional.l1_loss(
                    _m(_x.to(UD.DEVICE), _c.to(UD.DEVICE)).squeeze(-1), _y.to(UD.DEVICE))
                _tot += _l.item() * len(_y); _n += len(_y)
        _va = _tot / max(_n, 1)
        if _va < _best - 1e-6:
            _best, _bstate, _bad = _va, {k: v.clone() for k, v in _m.state_dict().items()}, 0
        else:
            _bad += 1
        if _bad >= 8:
            break
    if _bstate:
        _m.load_state_dict(_bstate)
    _m.eval()
    _rows = []
    with _torch.no_grad():
        for _x, _c, _y, _s, _ti in UD.DataLoader(_dsets[2], batch_size=1024):
            _o = _m(_x.to(UD.DEVICE), _c.to(UD.DEVICE)).squeeze(-1).cpu().numpy()
            _yh = _o * _dsets[2].ysd + _dsets[2].ymu
            _yt = _y.numpy() * _dsets[2].ysd + _dsets[2].ymu
            for _j, _h in enumerate(UP.HORIZONS):
                _rows.append(_pd.DataFrame({
                    UP.DATE: U_CAL.dates[_ti.numpy() + _h],
                    UP.STATE: [U_CAL.states[k] for k in _s.numpy()],
                    'target': _t, 'horizon': _h,
                    'y': _yt[:, _j], 'yhat': _yh[:, _j]}))
    _p = _pd.concat(_rows, ignore_index=True)
    _keep = [_p[_p.horizon == _h].merge(U_GRID[(_t, _h)][[UP.DATE, UP.STATE]],
                                        on=[UP.DATE, UP.STATE], how='inner')
             for _h in UP.HORIZONS]
    _orig_pred.append(_pd.concat(_keep, ignore_index=True))
    print(f"    {_t:<24} trained  ({_time.perf_counter()-_t0:.0f}s)")

U_REG.add('PatchTST+MCAG(replica)', _pd.concat(_orig_pred, ignore_index=True),
          section='U6', notes='original architecture + original 25-feature input, '
                             'reproduced on the upgrade grid for reference')
U_REG.checkpoint()

_rep = U_SCORER.headline(U_REG.stacked(['PatchTST+MCAG(replica)', 'Persistence']))
print("\n  Replica vs persistence on the observable targets:\n")
print(_rep[['model', 'MASE', 'RMSSE', 'MAPE', 'sMAPE', 'avg_MAPE_floored_6t', 'n_obs']]
      .to_string(index=False))
print("\n  The replica reproduces the published pattern: a transformer given no lag of")
print("  the target it must forecast cannot anchor the level, and lands far above a")
print("  one-line persistence rule. The architecture was never the bottleneck.")

In [ ]:
# ─── Cell U6.2 — MCAG-v2 base + the six extensions ─────────────────────────
print("=" * 78)
print("  U6.2 — MCAG-v2 VARIANTS")
print("=" * 78)

U_MCAG_VARIANTS = {
    'MCAGv2-base':      dict(),
    'MCAGv2-A-state':   dict(state_emb=True),
    'MCAGv2-B-regime':  dict(regime_emb=True),
    'MCAGv2-C-xstate':  dict(cross_state=True),
    'MCAGv2-D-fusion':  dict(fusion=True),
    'MCAGv2-E-multitask': dict(multitask=-1),   # -1 = "as many heads as there are
                                                # auxiliary targets for this target"
}

_AUX = ['grid_stress_index', 'avg_market_price']
U_MCAG_CFG = UD.TrainConfig(epochs=40, batch_size=512, lr=1e-3, patience=8,
                            loss='l1', seed=42)


def u_mcag_flags(flags, n_aux):
    """Resolve `multitask=-1` to the number of auxiliary targets available.

    The auxiliary set is `_AUX` minus the primary target, so it has 2 members for
    most targets but only 1 when the primary target is itself in `_AUX`. Fixing
    the head count at 2 would leave the multi-task head predicting a broadcast
    copy of a single auxiliary series on those targets, so the count is derived
    per target instead.
    """
    out = dict(flags)
    if out.get('multitask') == -1:
        out['multitask'] = n_aux
    return out

_mcag_pred = {k: [] for k in U_MCAG_VARIANTS}
_mcag_val = {k: [] for k in U_MCAG_VARIANTS}
U_MCAG_VAL = {}
_t0 = _time.perf_counter()

for _t in UP.TARGETS:
    _aux = [a for a in _AUX if a != _t]
    _ds = {}
    for _sp in ('train', 'valid', 'test'):
        _ds[_sp] = UM.MCAGWindows(U_CAL, _t, 28, UP.HORIZONS, _sp,
                                  regime_cols=UP.REGIME_COLS, aux_targets=_aux)
    _nch = _ds['train'].n_channels
    for _name, _flags in U_MCAG_VARIANTS.items():
        _cfg = UM.MCAGv2Config(lookback=28, n_out=len(UP.HORIZONS), d=64,
                               n_states=len(U_CAL.states),
                               n_regimes=len(UP.REGIME_COLS),
                               **u_mcag_flags(_flags, len(_aux)))
        UD.set_seed(42)
        _model = UM.MCAGv2(_cfg, _nch, _ds['train'].n_carbon, _ds['train'].n_price)
        _model, _hist, _bestval = UM.train_mcag(_model, _ds['train'], _ds['valid'],
                                                U_MCAG_CFG)
        U_MCAG_VAL[(_name, _t)] = _bestval
        _p = UM.predict_mcag(_model, _ds['test'], U_CAL.dates, U_CAL.states, _t,
                             UP.HORIZONS)
        _keep = [_p[_p.horizon == _h].merge(U_GRID[(_t, _h)][[UP.DATE, UP.STATE]],
                                            on=[UP.DATE, UP.STATE], how='inner')
                 for _h in UP.HORIZONS]
        _mcag_pred[_name].append(_pd.concat(_keep, ignore_index=True))
        _pv = UM.predict_mcag(_model, _ds['valid'], U_CAL.dates, U_CAL.states, _t,
                              UP.HORIZONS)
        _mcag_val[_name].append(_pd.concat(
            [_pv[_pv.horizon == _h].merge(U_GRID_VAL[(_t, _h)][[UP.DATE, UP.STATE]],
                                          on=[UP.DATE, UP.STATE], how='inner')
             for _h in UP.HORIZONS], ignore_index=True))
        del _model
    print(f"    {_t:<24} 6 variants done  ({_time.perf_counter()-_t0:.0f}s elapsed)")

for _name in U_MCAG_VARIANTS:
    U_REG.add(_name, _pd.concat(_mcag_pred[_name], ignore_index=True), section='U6',
              notes='carbon-gated PatchTST on instance-normalised windows')
    U_REG_VAL.add(_name, _pd.concat(_mcag_val[_name], ignore_index=True), section='U6')
U_REG.checkpoint()
U_REG_VAL.checkpoint('_registry_valid.pkl')
print(f"\n  trained {len(UP.TARGETS)*len(U_MCAG_VARIANTS)} models in {_time.perf_counter()-_t0:.0f}s")

In [ ]:
# ─── Cell U6.3 — Variant F: best combination, selected on VALIDATION ───────
print("=" * 78)
print("  U6.3 — VARIANT F: BEST PRACTICAL COMBINATION")
print("=" * 78)
print("  Extensions are ranked by their mean VALIDATION loss across the six targets;")
print("  every extension that improved on the base is switched on together. The test")
print("  split plays no part in this choice.\n")

_val_tbl = _pd.DataFrame([{'variant': k, 'mean_valid_loss': _np.mean(
    [U_MCAG_VAL[(k, t)] for t in UP.TARGETS])} for k in U_MCAG_VARIANTS])
_val_tbl = _val_tbl.sort_values('mean_valid_loss')
_base_val = float(_val_tbl.set_index('variant').loc['MCAGv2-base', 'mean_valid_loss'])
_val_tbl['vs_base'] = (_val_tbl['mean_valid_loss'] - _base_val) / _base_val * 100
print(_val_tbl.round(5).to_string(index=False))

_FLAG_OF = {'MCAGv2-A-state': 'state_emb', 'MCAGv2-B-regime': 'regime_emb',
            'MCAGv2-C-xstate': 'cross_state', 'MCAGv2-D-fusion': 'fusion'}
_helped = [k for k in _FLAG_OF
           if float(_val_tbl.set_index('variant').loc[k, 'mean_valid_loss']) < _base_val]
_combo = {_FLAG_OF[k]: True for k in _helped}
_mt_helped = float(_val_tbl.set_index('variant').loc['MCAGv2-E-multitask',
                                                     'mean_valid_loss']) < _base_val
if _mt_helped:
    _combo['multitask'] = -1        # resolved per target by u_mcag_flags
print(f"\n  extensions that beat the base on validation: {_helped}"
      f"{' + multitask' if _mt_helped else ''}")
print(f"  variant F configuration: {_combo if _combo else '(none helped -> F == base)'}")

_f_pred = []
_t0 = _time.perf_counter()
for _t in UP.TARGETS:
    _aux = [a for a in _AUX if a != _t]
    _ds = {sp: UM.MCAGWindows(U_CAL, _t, 28, UP.HORIZONS, sp,
                              regime_cols=UP.REGIME_COLS, aux_targets=_aux)
           for sp in ('train', 'valid', 'test')}
    _cfg = UM.MCAGv2Config(lookback=28, n_out=len(UP.HORIZONS), d=64,
                           n_states=len(U_CAL.states), n_regimes=len(UP.REGIME_COLS),
                           **u_mcag_flags(_combo, len(_aux)))
    UD.set_seed(42)
    _model = UM.MCAGv2(_cfg, _ds['train'].n_channels, _ds['train'].n_carbon,
                       _ds['train'].n_price)
    _model, _hist, _bv = UM.train_mcag(_model, _ds['train'], _ds['valid'], U_MCAG_CFG)
    _p = UM.predict_mcag(_model, _ds['test'], U_CAL.dates, U_CAL.states, _t, UP.HORIZONS)
    _keep = [_p[_p.horizon == _h].merge(U_GRID[(_t, _h)][[UP.DATE, UP.STATE]],
                                        on=[UP.DATE, UP.STATE], how='inner')
             for _h in UP.HORIZONS]
    _f_pred.append(_pd.concat(_keep, ignore_index=True))
    del _model
U_REG.add('MCAGv2-F-best', _pd.concat(_f_pred, ignore_index=True), section='U6',
          notes=f'validation-selected combination: {_combo}')
U_REG.checkpoint()
print(f"  variant F trained ({_time.perf_counter()-_t0:.0f}s)")

In [ ]:
# ─── Cell U6.4 — Common evaluation of the MCAG family ──────────────────────
print("=" * 78)
print("  U6.4 — MCAG-v2 EVALUATION")
print("=" * 78)

U_MCAG_NAMES = list(U_MCAG_VARIANTS) + ['MCAGv2-F-best']
_cmp = U_MCAG_NAMES + ['PatchTST+MCAG(replica)', 'Persistence']
_mc_stack = U_REG.stacked(_cmp)
_mc_head = U_SCORER.headline(_mc_stack)

print("\n  Headline — observable targets, aligned grid:\n")
print(_mc_head[['model', 'MASE', 'RMSSE', 'MAE', 'RMSE', 'MAPE', 'sMAPE',
                'avg_MAPE_floored_6t', 'n_obs']].to_string(index=False))

_pm = float(_mc_head.set_index('model').loc['Persistence', 'MASE'])
_bm = float(_mc_head.set_index('model').loc['MCAGv2-base', 'MASE'])
print(f"\n  Persistence  MASE = {_pm:.4f}")
print(f"  MCAGv2-base  MASE = {_bm:.4f}")
print("\n  Effect of each extension on TEST (negative = improvement over the v2 base):")
for _n in U_MCAG_NAMES:
    if _n == 'MCAGv2-base':
        continue
    _v = float(_mc_head.set_index('model').loc[_n, 'MASE'])
    print(f"    {_n:<22} {_v:.4f}   ({(_v-_bm)/_bm*100:+6.2f}% vs base)"
          f"   {'beats' if _v < _pm else 'does not beat'} persistence")

_mth = U_SCORER.by_target_horizon(_mc_stack)
_mc_head.to_csv(U_OUT / 'tables' / 'u6_mcag_headline.csv', index=False)
_mth.to_csv(U_OUT / 'tables' / 'u6_mcag_by_target_horizon.csv', index=False)
_val_tbl.to_csv(U_OUT / 'tables' / 'u6_mcag_validation_selection.csv', index=False)

_fig, _axes = _plt.subplots(1, 2, figsize=(14, 4.6))
_ax = _axes[0]
_d = _mc_head.set_index('model').reindex(_cmp)['MASE']
_cols = ['#e08214' if m.startswith('MCAGv2') else '#8a8a8a' for m in _d.index]
_ax.bar(range(len(_d)), _d.values, color=_cols)
_ax.axhline(_pm, color='crimson', ls='--', lw=1.3)
_ax.set_xticks(range(len(_d)))
_ax.set_xticklabels(_d.index, rotation=40, ha='right', fontsize=7)
_ax.set_ylabel('MASE'); _ax.set_title('(a) MCAG-v2 variants', fontweight='bold', fontsize=10)
_ax.grid(axis='y', alpha=0.3)

_ax = _axes[1]
_vt = _val_tbl.set_index('variant')['vs_base'].drop('MCAGv2-base')
_ax.barh(_vt.index, _vt.values,
         color=['#1a7f37' if v < 0 else '#c0392b' for v in _vt.values])
_ax.axvline(0, color='black', lw=1)
_ax.set_xlabel('validation loss vs base (%)  — negative = extension helped')
_ax.set_title('(b) Validation-only extension selection', fontweight='bold', fontsize=10)
_ax.tick_params(axis='y', labelsize=8); _ax.grid(axis='x', alpha=0.3)

_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U6_mcag_v2.png', dpi=150, bbox_inches='tight')
print(f"\n  saved -> {U_OUT / 'figures' / 'U6_mcag_v2.png'}")
_plt.show()

---
## Section U7 — Cross-State / State-Aware Forecasting

An 18-state panel offers a genuine choice: train one model on everything, or
one model per state. Pooling buys sample size and lets a state borrow strength
from its neighbours; separating buys specialisation. This section measures the
trade-off directly, holding everything else fixed.

| | Configuration |
|---|---|
| A | **pooled global** — one model for all states, no state identifier at all |
| B | **state-specific** — 18 independent models, each trained on its own state |
| C | **global + state** — pooled, with a state identifier |
| D | **global + state + regime** — pooled, with state identifier and the six train-only regime flags |

Every configuration uses the same architecture (LightGBM), the same features,
the same protocol and the same MASE-normalised persistence-residual target as
the winning Section U4 hybrid, so the only thing varying is state-awareness.

Results are reported three ways, because they can disagree: **overall pooled**,
**macro-averaged across states** (the headline), and **state-by-state**.

In [ ]:
# ─── Cell U7.1 — Four state-awareness configurations ───────────────────────
print("=" * 78)
print("  U7.1 — POOLED vs STATE-SPECIFIC vs STATE-AWARE")
print("=" * 78)

U_XS_CONFIGS = {
    'A-pooled-global':      dict(state=False, regime=False, per_state=False),
    'B-state-specific':     dict(state=False, regime=False, per_state=True),
    'C-global+state':       dict(state=True,  regime=False, per_state=False),
    'D-global+state+regime': dict(state=True, regime=True,  per_state=False),
}

_xs_pred = {k: [] for k in U_XS_CONFIGS}
_xs_val = {k: [] for k in U_XS_CONFIGS}
_xs_diag = []
_t_start = _time.perf_counter()

for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        _feat0 = UF.build_features(U_CAL, _t, _h)
        _feat0 = u_attach_base(_feat0, U_BASE_WIDE[('Persistence', _t, _h)])
        _feat0['base_minus_lag1'] = _feat0['base'] - _feat0['lag_1']
        _feat0 = _feat0.merge(U_REGIME_MAP, on=[UP.DATE, UP.STATE], how='left')
        _all_cols = UF.feature_columns(_feat0)

        for _name, _cfg in U_XS_CONFIGS.items():
            _cols = [c for c in _all_cols if c not in UP.REGIME_COLS]
            if not _cfg['state']:
                _cols = [c for c in _cols if c != 'state_id']
            if _cfg['regime']:
                _cols = _cols + list(UP.REGIME_COLS)

            if not _cfg['per_state']:
                _scale = U_SCORER.scale_for(_feat0[UP.STATE].to_numpy(), _t)
                _fit, _yh, _te, _yv, _va = UT.fit_predict_normalized(
                    'lgb', _feat0, _cols, _scale,
                    base=_feat0['base'].to_numpy(float), seed=42)
                _xs_pred[_name].append(u_frame(_te, _yh, _t, _h))
                _xs_val[_name].append(u_frame(_va, _yv, _t, _h))
                _xs_diag.append({'config': _name, 'target': _t, 'horizon': _h,
                                 'n_models': 1, 'rounds': _fit.best_rounds})
            else:
                # 18 independent models, each seeing only its own state's history
                _tp, _vp, _rounds = [], [], []
                for _st, _g in _feat0.groupby(UP.STATE, sort=False):
                    _g = _g.reset_index(drop=True)
                    if (_g['split'] == 'train').sum() < 200:
                        continue
                    _sc1 = U_SCORER.scale_for(_g[UP.STATE].to_numpy(), _t)
                    _f2, _yh2, _te2, _yv2, _va2 = UT.fit_predict_normalized(
                        'lgb', _g, [c for c in _cols if c != 'state_id'], _sc1,
                        base=_g['base'].to_numpy(float), seed=42)
                    _tp.append(u_frame(_te2, _yh2, _t, _h))
                    _vp.append(u_frame(_va2, _yv2, _t, _h))
                    _rounds.append(_f2.best_rounds)
                _xs_pred[_name].append(_pd.concat(_tp, ignore_index=True))
                _xs_val[_name].append(_pd.concat(_vp, ignore_index=True))
                _xs_diag.append({'config': _name, 'target': _t, 'horizon': _h,
                                 'n_models': len(_tp),
                                 'rounds': float(_np.mean(_rounds)) if _rounds else _np.nan})
        del _feat0
    print(f"    {_t:<24} done  ({_time.perf_counter()-_t_start:.0f}s elapsed)")

for _name in U_XS_CONFIGS:
    U_REG.add(f'XS_{_name}', _pd.concat(_xs_pred[_name], ignore_index=True),
              section='U7', notes='persistence-residual LightGBM, state-awareness ablation')
    U_REG_VAL.add(f'XS_{_name}', _pd.concat(_xs_val[_name], ignore_index=True), section='U7')
U_XS_NAMES = [f'XS_{k}' for k in U_XS_CONFIGS]
U_REG.checkpoint(); U_REG_VAL.checkpoint('_registry_valid.pkl')
print(f"\n  done in {_time.perf_counter()-_t_start:.0f}s")

In [ ]:
# ─── Cell U7.2 — Overall, macro and state-wise evaluation ──────────────────
print("=" * 78)
print("  U7.2 — CROSS-STATE EVALUATION")
print("=" * 78)

_cmp = U_XS_NAMES + ['Persistence']
_xs_stack = U_REG.stacked(_cmp)
_xs_head = U_SCORER.headline(_xs_stack)

print("\n  [A] Headline — observable targets (MASE / RMSSE are macro-averaged")
print("      across the 18 states; *_pooled treat the panel as one series):\n")
print(_xs_head[['model', 'MASE', 'MASE_pooled', 'RMSSE', 'MAE', 'RMSE',
                'MAPE', 'sMAPE', 'n_obs']].to_string(index=False))

print("\n  Note how MASE (macro) and MASE_pooled rank the configurations differently —")
print("  pooling hides what happens in the smallest states, which is exactly the")
print("  effect Section U3.2 isolated.")

print("\n  [B] MASE by target x horizon:\n")
_xth = U_SCORER.by_target_horizon(_xs_stack)
print(_xth.pivot_table(index=['target', 'horizon'], columns='model',
                       values='MASE')[_cmp].round(4).to_string())

print("\n  [C] State-wise MASE, total_generation_mwh (h pooled):\n")
_xs_state = U_SCORER.by_state(_xs_stack, targets=['total_generation_mwh'])
_piv = _xs_state.pivot(index='state_name', columns='model', values='MASE')[_cmp]
_scale_med = U_DF[U_DF['_split'] == 'train'].groupby(UP.STATE)['total_generation_mwh'].median()
_piv.insert(0, 'train_median', _scale_med)
print(_piv.sort_values('train_median').round(4).to_string())

_bestcfg = str(_xs_head.iloc[0]['model'])
print(f"\n  BEST STATE-AWARENESS CONFIGURATION: {_bestcfg}  "
      f"(MASE {float(_xs_head.iloc[0]['MASE']):.4f})")

_xs_head.to_csv(U_OUT / 'tables' / 'u7_crossstate_headline.csv', index=False)
_xs_state.to_csv(U_OUT / 'tables' / 'u7_crossstate_by_state.csv', index=False)
_pd.DataFrame(_xs_diag).to_csv(U_OUT / 'tables' / 'u7_crossstate_diagnostics.csv', index=False)

_fig, _axes = _plt.subplots(1, 2, figsize=(15, 4.8))
_ax = _axes[0]
_w = 0.35
_x = _np.arange(len(_cmp))
_ax.bar(_x - _w/2, _xs_head.set_index('model').reindex(_cmp)['MASE'], _w,
        label='MASE (macro across states)', color='#2c7fb8')
_ax.bar(_x + _w/2, _xs_head.set_index('model').reindex(_cmp)['MASE_pooled'], _w,
        label='MASE (pooled panel)', color='#f0a202')
_ax.set_xticks(_x); _ax.set_xticklabels(_cmp, rotation=30, ha='right', fontsize=7.5)
_ax.set_ylabel('MASE'); _ax.legend(fontsize=8); _ax.grid(axis='y', alpha=0.3)
_ax.set_title('(a) Macro vs pooled scoring', fontweight='bold', fontsize=10)

_ax = _axes[1]
_pl = _piv.drop(columns='train_median').sort_index()
for _c in _pl.columns:
    _ax.plot(range(len(_pl)), _pl[_c].values, marker='o', ms=4, lw=1.4, label=_c)
_ax.set_xticks(range(len(_pl)))
_ax.set_xticklabels(_pl.index, rotation=75, fontsize=6.5)
_ax.set_ylabel('MASE'); _ax.set_yscale('log')
_ax.set_title('(b) State-wise, generation', fontweight='bold', fontsize=10)
_ax.legend(fontsize=7); _ax.grid(alpha=0.3)
_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U7_cross_state.png', dpi=150, bbox_inches='tight')
print(f"\n  saved -> {U_OUT / 'figures' / 'U7_cross_state.png'}")
_plt.show()

---
## Section U8 — Time-Series Foundation Models

Zero-shot foundation models (TimesFM, Chronos / Chronos-Bolt) are the obvious
recent comparison point. They are only worth including if they can be run
**without disturbing the environment that produced the original results** —
a torch downgrade or a numpy bump would silently invalidate every number in
Part I of this notebook.

So this section checks availability first and reports honestly:

* if a compatible model is importable, it is run zero-shot on the same grid;
* if not, the section prints **"Not available — skipped."** and moves on.

No result is ever fabricated, and no package is installed that would alter the
pinned `torch` / `numpy` / `pandas` versions.

In [ ]:
# ─── Cell U8.1 — Availability check ─────────────────────────────────────────
import importlib as _il

print("=" * 78)
print("  U8.1 — FOUNDATION MODEL AVAILABILITY")
print("=" * 78)

_PINNED = {'torch': _torch.__version__ if '_torch' in dir() else 'n/a',
           'numpy': _np.__version__, 'pandas': _pd.__version__}
print(f"  environment currently pinned at: {_PINNED}")
print("  a foundation model is only used if it imports WITHOUT changing these.\n")

U_FM_CANDIDATES = ['chronos', 'timesfm', 'momentfm', 'uni2ts']
U_FM_STATUS = {}
for _mod in U_FM_CANDIDATES:
    try:
        _il.import_module(_mod)
        U_FM_STATUS[_mod] = 'available'
    except Exception as _e:
        U_FM_STATUS[_mod] = f'not installed ({type(_e).__name__})'
    print(f"    {_mod:<12} {U_FM_STATUS[_mod]}")

U_FM_AVAILABLE = [k for k, v in U_FM_STATUS.items() if v == 'available']
print()
if not U_FM_AVAILABLE:
    print("  Not available — skipped.")
    print()
    print("  Rationale for not installing one here: `chronos-forecasting` and")
    print("  `timesfm` both pull a transformers/accelerate stack and pin their own")
    print("  torch build. This environment holds torch 2.5.1+cu121, which produced")
    print("  every result in Part I of this notebook. Silently resolving a different")
    print("  torch to add one zero-shot baseline would put the original")
    print("  reproducibility record at risk for a comparison that is not required")
    print("  by any claim made in Part II.")
    print()
    print("  To add it deliberately in an isolated environment:")
    print("      python -m venv .venv-fm && .venv-fm/Scripts/pip install chronos-forecasting")
    print("      # then score it on results/forecasting_upgrade/tables/ grids")
else:
    print(f"  Available: {U_FM_AVAILABLE} — running zero-shot below.")

_pd.DataFrame([{'model': k, 'status': v} for k, v in U_FM_STATUS.items()]).to_csv(
    U_OUT / 'tables' / 'u8_foundation_availability.csv', index=False)

In [ ]:
# ─── Cell U8.2 — Zero-shot run (only if a model is importable) ─────────────
U_FM_NAMES = []
if not U_FM_AVAILABLE:
    print("=" * 78)
    print("  U8.2 — Not available — skipped.")
    print("=" * 78)
    print("  No foundation-model row appears in the Section U14 leaderboard.")
    print("  This is recorded as a gap in the comparison, not as a negative result:")
    print("  nothing here licenses any claim about how TimesFM or Chronos would")
    print("  perform on this panel.")
else:
    print("=" * 78)
    print("  U8.2 — ZERO-SHOT FOUNDATION MODEL")
    print("=" * 78)
    _fm_pred = []
    if 'chronos' in U_FM_AVAILABLE:
        from chronos import ChronosPipeline
        _pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-small",
                                                device_map=str(UD.DEVICE),
                                                torch_dtype=_torch.float32)
        _ctx = 128
        for _t in UP.TARGETS:
            _yw = U_CAL.wide(_t)
            _rows = []
            for _st in U_CAL.states:
                _s = _yw[_st]
                for _h in UP.HORIZONS:
                    _g = U_GRID[(_t, _h)]
                    _g = _g[_g[UP.STATE] == _st]
                    for _, _r in _g.iterrows():
                        _origin = _r[UP.DATE] - _pd.Timedelta(days=_h)
                        _hist = _s.loc[:_origin].dropna().to_numpy()[-_ctx:]
                        if len(_hist) < 16:
                            continue
                        _fc = _pipe.predict(_torch.tensor(_hist, dtype=_torch.float32),
                                            prediction_length=_h, num_samples=20)
                        _rows.append({UP.DATE: _r[UP.DATE], UP.STATE: _st,
                                      'target': _t, 'horizon': _h, 'y': _r['y'],
                                      'yhat': float(_fc[0, :, -1].median())})
            _fm_pred.append(_pd.DataFrame(_rows))
        U_REG.add('Chronos-zeroshot', _pd.concat(_fm_pred, ignore_index=True),
                  section='U8', notes='zero-shot, context 128, median of 20 samples')
        U_FM_NAMES.append('Chronos-zeroshot')
        U_REG.checkpoint()
        print(U_SCORER.headline(U_REG.stacked(U_FM_NAMES + ['Persistence']))
              [['model', 'MASE', 'RMSSE', 'MAE', 'sMAPE', 'n_obs']].to_string(index=False))

---
## Section U9 — Ensembles

Candidates are the strongest model from each family in Sections U2–U7. Three
combination rules, in increasing order of how much they can overfit:

1. **Simple average** — equal weights, nothing estimated.
2. **Validation-weighted** — non-negative weights on the simplex, chosen to
   minimise the *validation* macro-MASE objective.
3. **Stacking** — a ridge meta-learner on the validation predictions.

**Every weight is estimated on the VALIDATION split only.** That is the whole
reason `U_REG_VAL` exists: each model's validation predictions come from the
early-stopping fit, which never saw the validation block, and the test split is
untouched until the final scoring line.

In [ ]:
# ─── Cell U9.1 — Candidate selection (on validation) ───────────────────────
from scipy.optimize import minimize as _minimize

print("=" * 78)
print("  U9.1 — ENSEMBLE CANDIDATES")
print("=" * 78)

# Only models with BOTH test and validation predictions can enter an ensemble.
_eligible = [m for m in U_REG.preds if m in U_REG_VAL.preds]
_val_head = U_SCORER.headline(U_REG_VAL.stacked(_eligible))
print("\n  Ranked by VALIDATION MASE (test not consulted):\n")
print(_val_head[['model', 'MASE', 'RMSSE', 'MAE', 'sMAPE']].head(14).to_string(index=False))

# One representative per family, plus persistence as the anchor.
_families = {
    'statistical': [m for m in _eligible if m in ('Persistence', 'ETS', 'ARIMA',
                                                  'RollingMean', 'SeasonalNaive', 'Drift')],
    'tree': [m for m in _eligible if m in ('LightGBM', 'XGBoost', 'CatBoost')],
    'hybrid': [m for m in _eligible if m.startswith('Hybrid_')],
    'deep': [m for m in _eligible if m in ('N-BEATS', 'N-HiTS', 'TSMixer')],
    'mcag': [m for m in _eligible if m.startswith('MCAGv2')],
    'crossstate': [m for m in _eligible if m.startswith('XS_')],
}
_rank = _val_head.set_index('model')['MASE']
U_ENS_CANDIDATES = []
for _fam, _members in _families.items():
    _members = [m for m in _members if m in _rank.index]
    if not _members:
        continue
    _best = min(_members, key=lambda m: _rank[m])
    U_ENS_CANDIDATES.append(_best)
    print(f"    {_fam:<12} -> {_best:<26} (valid MASE {_rank[_best]:.4f})")
U_ENS_CANDIDATES = list(dict.fromkeys(U_ENS_CANDIDATES))
if 'Persistence' not in U_ENS_CANDIDATES:
    U_ENS_CANDIDATES.append('Persistence')
print(f"\n  candidates: {U_ENS_CANDIDATES}")

In [ ]:
# ─── Cell U9.2 — Build the three ensembles ─────────────────────────────────
print("=" * 78)
print("  U9.2 — COMBINATION RULES")
print("=" * 78)

_KEYS = [UP.DATE, UP.STATE, 'target', 'horizon']


def _wide_preds(reg, names):
    """(keys + y) x one column per model, on the models' common rows."""
    base = None
    for n in names:
        p = reg.preds[n][_KEYS + ['y', 'yhat']].rename(columns={'yhat': n})
        base = p if base is None else base.merge(p.drop(columns='y'), on=_KEYS, how='inner')
    return base


_val_w = _wide_preds(U_REG_VAL, U_ENS_CANDIDATES)
_test_w = _wide_preds(U_REG, U_ENS_CANDIDATES)
print(f"  aligned rows — validation {len(_val_w):,}   test {len(_test_w):,}")

# Errors are divided by the per-(state,target) MASE denominator so the objective
# being minimised IS the macro-MASE the leaderboard reports.
def _scale_col(df):
    return _np.array([U_SCORER.scales.get((s, t), 1.0)
                      for s, t in zip(df[UP.STATE], df['target'])], dtype=float)


_sv = _scale_col(_val_w)
_Pv = _val_w[U_ENS_CANDIDATES].to_numpy(float)
_yv = _val_w['y'].to_numpy(float)
_st_ = _scale_col(_test_w)
_Pt = _test_w[U_ENS_CANDIDATES].to_numpy(float)

# --- 1. simple average ----------------------------------------------------
_w_avg = _np.ones(len(U_ENS_CANDIDATES)) / len(U_ENS_CANDIDATES)

# --- 2. validation-weighted (non-negative, sums to 1) ---------------------
def _obj(w):
    return float(_np.mean(_np.abs(_Pv @ w - _yv) / _sv))


_res = _minimize(_obj, _w_avg, method='SLSQP',
                 bounds=[(0.0, 1.0)] * len(U_ENS_CANDIDATES),
                 constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
                 options={'maxiter': 300, 'ftol': 1e-10})
_w_opt = _res.x / _res.x.sum()

# --- 3. stacking: ridge meta-learner on validation ------------------------
from sklearn.linear_model import Ridge as _Ridge
_Zv = _Pv / _sv[:, None]
_Zt = _Pt / _st_[:, None]
_stack = _Ridge(alpha=1.0, fit_intercept=True).fit(_Zv, _yv / _sv)

print("\n  Learned weights (validation only):\n")
print(f"    {'model':<28} {'simple':>8} {'val-weighted':>14} {'stack coef':>12}")
for _i, _m in enumerate(U_ENS_CANDIDATES):
    print(f"    {_m:<28} {_w_avg[_i]:>8.3f} {_w_opt[_i]:>14.3f} {_stack.coef_[_i]:>12.3f}")
print(f"    {'(intercept)':<28} {'':>8} {'':>14} {_stack.intercept_:>12.3f}")

for _nm, _yhat in [
    ('Ensemble-SimpleAvg', _Pt @ _w_avg),
    ('Ensemble-ValWeighted', _Pt @ _w_opt),
    ('Ensemble-Stacked', _stack.predict(_Zt) * _st_),
]:
    _pr = _test_w[_KEYS + ['y']].copy()
    _pr['yhat'] = _yhat
    U_REG.add(_nm, _pr, section='U9', notes=f'candidates={U_ENS_CANDIDATES}')
U_ENS_NAMES = ['Ensemble-SimpleAvg', 'Ensemble-ValWeighted', 'Ensemble-Stacked']
U_REG.checkpoint()

In [ ]:
# ─── Cell U9.3 — Ensemble evaluation ───────────────────────────────────────
print("=" * 78)
print("  U9.3 — ENSEMBLE EVALUATION")
print("=" * 78)

_cmp = U_ENS_NAMES + U_ENS_CANDIDATES
_ens_stack = U_REG.stacked(_cmp)
_ens_head = U_SCORER.headline(_ens_stack)
print()
print(_ens_head[['model', 'MASE', 'RMSSE', 'MAE', 'RMSE', 'MAPE', 'sMAPE',
                 'avg_MAPE_floored_6t', 'n_obs']].to_string(index=False))

_best_single = min([m for m in U_ENS_CANDIDATES],
                   key=lambda m: float(_ens_head.set_index('model').loc[m, 'MASE']))
_bs = float(_ens_head.set_index('model').loc[_best_single, 'MASE'])
print(f"\n  best single candidate: {_best_single} (MASE {_bs:.4f})")
for _e in U_ENS_NAMES:
    _v = float(_ens_head.set_index('model').loc[_e, 'MASE'])
    print(f"    {_e:<24} {_v:.4f}   ({(_v-_bs)/_bs*100:+6.2f}% vs best single)")

_ens_head.to_csv(U_OUT / 'tables' / 'u9_ensemble_headline.csv', index=False)
_pd.DataFrame({'model': U_ENS_CANDIDATES, 'w_simple': _w_avg,
               'w_validation': _w_opt, 'stack_coef': _stack.coef_}).to_csv(
    U_OUT / 'tables' / 'u9_ensemble_weights.csv', index=False)

---
## Section U10 — Probabilistic Forecasting & Uncertainty

The existing quantile head (Cell 3f) is **under-covered**: its nominal 80%
interval achieves 63.7% on `total_generation_mwh` and 63.4% on `log_carbon`
(`results/tables/metrics_summary.json → probabilistic_forecasting`). An
interval that claims 80% and delivers 64% is not a calibration detail — it
understates risk by a wide margin, which matters for anything downstream that
sizes a reserve or a hedge.

Three approaches, all calibrated on **validation only**:

1. **Quantile regression** — LightGBM pinball loss at τ ∈ {0.1, 0.5, 0.9},
   trained on the MASE-normalised persistence residual.
2. **Split conformal prediction** — take the residual quantiles of the
   validation set and widen the interval by exactly the amount needed for
   nominal coverage. Distribution-free, and finite-sample valid under
   exchangeability.
3. **Conformalised quantile regression (CQR)** — the standard combination:
   quantile regression for the shape, a conformal correction for the coverage.

Reported: CRPS, PI80 coverage, mean interval width, WQL, and MAE/RMSE of the
P50 forecast.

In [ ]:
# ─── Cell U10.1 — Quantile regression + conformal calibration ──────────────
import lightgbm as _lgb

print("=" * 78)
print("  U10.1 — QUANTILE MODELS")
print("=" * 78)

U_TAUS = [0.1, 0.5, 0.9]
_q_rows, _q_val_rows = [], []
_t_start = _time.perf_counter()

for _t in UP.TARGETS:
    for _h in UP.HORIZONS:
        _feat = UF.build_features(U_CAL, _t, _h)
        _feat = u_attach_base(_feat, U_BASE_WIDE[('Persistence', _t, _h)])
        _feat['base_minus_lag1'] = _feat['base'] - _feat['lag_1']
        _fcols = UF.feature_columns(_feat)
        _scale = U_SCORER.scale_for(_feat[UP.STATE].to_numpy(), _t)
        _feat['_z'] = (_feat['y'] - _feat['base']) / _scale
        _feat['_scale'] = _scale

        _tr = _feat[_feat.split == 'train']
        _va = _feat[_feat.split == 'valid'].reset_index(drop=True)
        _te = _feat[_feat.split == 'test'].reset_index(drop=True)

        _qv, _qt = {}, {}
        for _tau in U_TAUS:
            _p = dict(objective='quantile', alpha=_tau, learning_rate=0.06,
                      num_leaves=63, min_data_in_leaf=40, feature_fraction=0.85,
                      bagging_fraction=0.85, bagging_freq=1, verbosity=-1, seed=42)
            _dtr = _lgb.Dataset(_tr[_fcols], _tr['_z'])
            _dva = _lgb.Dataset(_va[_fcols], _va['_z'], reference=_dtr)
            _b = _lgb.train(_p, _dtr, num_boost_round=UT.MAX_ROUNDS, valid_sets=[_dva],
                            callbacks=[_lgb.early_stopping(UT.EARLY_STOP, verbose=False)])
            _best = int(_b.best_iteration or UT.MAX_ROUNDS)
            _qv[_tau] = _b.predict(_va[_fcols], num_iteration=_best)
            _full = _lgb.Dataset(_pd.concat([_tr[_fcols], _va[_fcols]]),
                                 _pd.concat([_tr['_z'], _va['_z']]))
            _bf = _lgb.train(_p, _full, num_boost_round=_best)
            _qt[_tau] = _bf.predict(_te[_fcols])

        # back to original units
        def _un(z, rows):
            return _np.asarray(z) * rows['_scale'].to_numpy() + rows['base'].to_numpy()

        _q_val_rows.append(_pd.DataFrame({
            UP.DATE: _va[UP.DATE].to_numpy(), UP.STATE: _va[UP.STATE].to_numpy(),
            'target': _t, 'horizon': _h, 'y': _va['y'].to_numpy(float),
            **{f'q{int(_tau*100)}': _un(_qv[_tau], _va) for _tau in U_TAUS}}))
        _q_rows.append(_pd.DataFrame({
            UP.DATE: _te[UP.DATE].to_numpy(), UP.STATE: _te[UP.STATE].to_numpy(),
            'target': _t, 'horizon': _h, 'y': _te['y'].to_numpy(float),
            **{f'q{int(_tau*100)}': _un(_qt[_tau], _te) for _tau in U_TAUS}}))
        del _feat
    print(f"    {_t:<24} done  ({_time.perf_counter()-_t_start:.0f}s elapsed)")

U_QVAL = _pd.concat(_q_val_rows, ignore_index=True)
U_QTEST = _pd.concat(_q_rows, ignore_index=True)
print(f"\n  quantile models fitted in {_time.perf_counter()-_t_start:.0f}s")

In [ ]:
# ─── Cell U10.2 — Conformal calibration on the validation split ────────────
print("=" * 78)
print("  U10.2 — CONFORMAL CALIBRATION (validation only)")
print("=" * 78)
print("  Split conformal: for nominal coverage 1-a, take the (1-a) empirical")
print("  quantile of the validation nonconformity scores and widen the interval by")
print("  exactly that much. CQR uses the score max(q_lo - y, y - q_hi), which")
print("  widens an interval that is too narrow and SHRINKS one that is too wide.\n")

_ALPHA = 0.20        # nominal 80% interval, matching the existing Cell 3f report
U_CONF = {}
for (_t, _h), _g in U_QVAL.groupby(['target', 'horizon']):
    # CQR nonconformity score
    _s = _np.maximum(_g['q10'].to_numpy() - _g['y'].to_numpy(),
                     _g['y'].to_numpy() - _g['q90'].to_numpy())
    _n = len(_s)
    _k = min(_n, int(_np.ceil((_n + 1) * (1 - _ALPHA))))
    _qhat = float(_np.sort(_s)[_k - 1]) if _n else 0.0
    # absolute-residual score for the simpler symmetric conformal interval
    _r = _np.abs(_g['y'].to_numpy() - _g['q50'].to_numpy())
    _rhat = float(_np.sort(_r)[min(_n, int(_np.ceil((_n + 1) * (1 - _ALPHA)))) - 1]) if _n else 0.0
    U_CONF[(_t, _h)] = {'cqr_qhat': _qhat, 'sym_qhat': _rhat, 'n_calib': _n}

_cf = _pd.DataFrame([{'target': k[0], 'horizon': k[1], **v} for k, v in U_CONF.items()])
print(_cf.pivot_table(index='target', columns='horizon',
                      values='cqr_qhat').round(3).to_string())
print("\n  positive q-hat = the validation intervals were too NARROW and are widened;")
print("  negative q-hat = they were too wide and are tightened.")

U_QTEST['cqr_lo'] = U_QTEST['q10'] - U_QTEST.set_index(['target', 'horizon']).index.map(
    lambda k: U_CONF[k]['cqr_qhat'])
U_QTEST['cqr_hi'] = U_QTEST['q90'] + U_QTEST.set_index(['target', 'horizon']).index.map(
    lambda k: U_CONF[k]['cqr_qhat'])
U_QTEST['sym_lo'] = U_QTEST['q50'] - U_QTEST.set_index(['target', 'horizon']).index.map(
    lambda k: U_CONF[k]['sym_qhat'])
U_QTEST['sym_hi'] = U_QTEST['q50'] + U_QTEST.set_index(['target', 'horizon']).index.map(
    lambda k: U_CONF[k]['sym_qhat'])

In [ ]:
# ─── Cell U10.3 — Probabilistic evaluation ─────────────────────────────────
print("=" * 78)
print("  U10.3 — PROBABILISTIC METRICS  (nominal 80% interval)")
print("=" * 78)

from refused_fixes.metrics import crps_from_quantiles as _crps

_rows = []
for _t, _g in U_QTEST.groupby('target'):
    _y = _g['y'].to_numpy(float)
    for _label, _lo, _hi in [('QuantileReg', 'q10', 'q90'),
                             ('Conformal-sym', 'sym_lo', 'sym_hi'),
                             ('CQR', 'cqr_lo', 'cqr_hi')]:
        _l, _hgh = _g[_lo].to_numpy(float), _g[_hi].to_numpy(float)
        _cov = float(_np.mean((_y >= _l) & (_y <= _hgh)))
        _wid = float(_np.mean(_hgh - _l))
        _qp = _np.stack([_l, _g['q50'].to_numpy(float), _hgh], axis=1)
        _rows.append({
            'target': _t, 'method': _label,
            'PI80_coverage': round(_cov, 4),
            'coverage_gap_pp': round((_cov - 0.80) * 100, 2),
            'mean_width': round(_wid, 4),
            'CRPS': round(_crps(_y, _qp, U_TAUS), 4),
            'WQL': round(UE.wql(_y, _qp, U_TAUS), 5),
            'P50_MAE': round(UE.mae(_y, _g['q50']), 4),
            'P50_RMSE': round(UE.rmse(_y, _g['q50']), 4),
        })
U_PROB = _pd.DataFrame(_rows)
print()
print(U_PROB.to_string(index=False))

_EXISTING = {'total_generation_mwh': 0.6368, 'avg_market_price': 0.8227,
             'grid_stress_index': 0.7664, 'fcfs_priority_score': 0.6679,
             'palmp': 0.8371, 'log_carbon': 0.6343}
def _cov_of(target, method):
    """PI80 coverage for one (target, method) cell of U_PROB."""
    _s = U_PROB[(U_PROB.target == target) & (U_PROB.method == method)]['PI80_coverage']
    return float(_s.iloc[0]) if len(_s) else _np.nan


print("\n  Coverage vs the EXISTING Cell 3f quantile head (nominal 0.80):\n")
print(f"    {'target':<24} {'existing':>9} {'QuantileReg':>12} {'CQR':>9}   verdict")
for _t in UP.TARGETS:
    _ex = _EXISTING.get(_t, _np.nan)
    _qr = _cov_of(_t, 'QuantileReg')
    _cq = _cov_of(_t, 'CQR')
    _v = 'closer to nominal' if abs(_cq - 0.8) < abs(_ex - 0.8) else 'not improved'
    print(f"    {_t:<24} {_ex:>9.4f} {_qr:>12.4f} {_cq:>9.4f}   {_v}")

_ok = sum(abs(_cov_of(t, 'CQR') - 0.8) < abs(_EXISTING.get(t, 1) - 0.8)
          for t in UP.TARGETS)
print(f"\n  CQR is closer to nominal coverage on {_ok}/{len(UP.TARGETS)} targets.")
print("  Split conformal guarantees coverage only under EXCHANGEABILITY between the")
print("  calibration block (validation) and the test block. That assumption is")
print("  violated on this panel wherever the series shifts between 2023 and 2024-25,")
print("  so the guarantee is not automatic here and is checked rather than assumed:")

_miscovered = []
for _t in UP.TARGETS:
    _cq = _cov_of(_t, 'CQR')
    if _np.isfinite(_cq) and abs(_cq - 0.80) > 0.05:
        _miscovered.append((_t, _cq))
if _miscovered:
    for _t, _cq in _miscovered:
        print(f"    {_t:<24} CQR coverage {_cq:.4f} vs nominal 0.80 "
              f"({(_cq-0.80)*100:+.1f} pp) — calibration did NOT transfer")
    print("    On these targets the validation-fitted q-hat is the wrong width for the")
    print("    test period; the interval must not be reported as an 80% interval.")
    print("    This is the same train->test shift Section U1 measured, not a coding error.")
else:
    print("    All targets land within 5 pp of nominal: calibration transferred.")
print("  Coverage must always be quoted together with the interval WIDTH above —")
print("  an interval can buy coverage simply by becoming uninformatively wide.")

U_PROB.to_csv(U_OUT / 'tables' / 'u10_probabilistic.csv', index=False)
_cf.to_csv(U_OUT / 'tables' / 'u10_conformal_qhat.csv', index=False)

_fig, _axes = _plt.subplots(1, 2, figsize=(14, 4.6))
_ax = _axes[0]
_x = _np.arange(len(UP.TARGETS)); _w = 0.26
for _i, (_m, _c) in enumerate([('QuantileReg', '#2c7fb8'), ('Conformal-sym', '#f0a202'),
                               ('CQR', '#1a7f37')]):
    _v = [_cov_of(t, _m) for t in UP.TARGETS]
    _ax.bar(_x + (_i - 1) * _w, _v, _w, label=_m, color=_c)
_ax.plot(_x, [_EXISTING.get(t, _np.nan) for t in UP.TARGETS], 'kv', ms=8, label='existing Cell 3f')
_ax.axhline(0.80, color='crimson', ls='--', lw=1.4)
_ax.set_xticks(_x); _ax.set_xticklabels([t[:14] for t in UP.TARGETS], rotation=35,
                                        ha='right', fontsize=7.5)
_ax.set_ylabel('PI80 empirical coverage'); _ax.set_ylim(0, 1.05)
_ax.set_title('(a) Coverage vs nominal 0.80', fontweight='bold', fontsize=10)
_ax.legend(fontsize=7.5); _ax.grid(axis='y', alpha=0.3)

_ax = _axes[1]
_g = U_QTEST[(U_QTEST.target == 'total_generation_mwh') & (U_QTEST.horizon == 1) &
             (U_QTEST[UP.STATE] == 'Maharashtra')].sort_values(UP.DATE).head(150)
_ax.fill_between(_g[UP.DATE], _g['cqr_lo'], _g['cqr_hi'], alpha=0.25,
                 color='#1a7f37', label='CQR 80%')
_ax.plot(_g[UP.DATE], _g['q50'], lw=1.3, color='#1a7f37', label='P50')
_ax.plot(_g[UP.DATE], _g['y'], lw=1.1, color='black', label='actual')
_ax.set_title('(b) Calibrated interval — Maharashtra, h=1', fontweight='bold', fontsize=10)
_ax.set_ylabel('generation (GWh/day)'); _ax.legend(fontsize=8); _ax.grid(alpha=0.3)
_plt.setp(_ax.get_xticklabels(), rotation=30, ha='right', fontsize=7)
_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U10_probabilistic.png', dpi=150, bbox_inches='tight')
print(f"\n  saved -> {U_OUT / 'figures' / 'U10_probabilistic.png'}")
_plt.show()

---
## Section U11 — Hyperparameters & Multi-Seed Robustness

Two questions about the leading candidates, both answered without touching the
test split for selection:

1. **Does the result depend on hyperparameters I happened to pick?**
   A small grid is searched on VALIDATION only.
2. **Does it survive re-seeding?** The strongest candidates are retrained under
   several seeds and reported as mean ± std with a 95% confidence interval.

The seed is *never* chosen by test performance. A single-seed improvement that
vanishes across seeds is not a result, and this section is what decides which
of those two the upgrade has.

In [ ]:
# ─── Cell U11.1 — Validation-only hyperparameter search ────────────────────
print("=" * 78)
print("  U11.1 — HYPERPARAMETER SEARCH (validation only)")
print("=" * 78)

U_HP_GRID = [
    dict(num_leaves=31,  learning_rate=0.06, min_data_in_leaf=40),
    dict(num_leaves=63,  learning_rate=0.06, min_data_in_leaf=40),
    dict(num_leaves=63,  learning_rate=0.03, min_data_in_leaf=20),
    dict(num_leaves=127, learning_rate=0.06, min_data_in_leaf=60),
    dict(num_leaves=31,  learning_rate=0.10, min_data_in_leaf=20),
]

_hp_rows = []
_t_start = _time.perf_counter()
for _gi, _grid in enumerate(U_HP_GRID):
    _vals = []
    for _t in UP.OBSERVABLE_TARGETS:          # search on the headline targets
        for _h in UP.HORIZONS:
            _feat = UF.build_features(U_CAL, _t, _h)
            _feat = u_attach_base(_feat, U_BASE_WIDE[('Persistence', _t, _h)])
            _feat['base_minus_lag1'] = _feat['base'] - _feat['lag_1']
            _fc = UF.feature_columns(_feat)
            _sc = U_SCORER.scale_for(_feat[UP.STATE].to_numpy(), _t)
            _fit, _yh, _te, _yv, _va = UT.fit_predict_normalized(
                'lgb', _feat, _fc, _sc, base=_feat['base'].to_numpy(float),
                params=_grid, seed=42)
            _vals.append(U_SCORER.score_group(u_frame(_va, _yv, _t, _h), _t)['MASE'])
            del _feat
    _hp_rows.append({**_grid, 'valid_MASE': float(_np.mean(_vals))})
    print(f"    grid {_gi+1}/{len(U_HP_GRID)} {_grid} -> valid MASE "
          f"{_np.mean(_vals):.4f}   ({_time.perf_counter()-_t_start:.0f}s)")

U_HP = _pd.DataFrame(_hp_rows).sort_values('valid_MASE').reset_index(drop=True)
U_BEST_HP = {k: v for k, v in U_HP.iloc[0].items() if k != 'valid_MASE'}
U_BEST_HP = {k: (int(v) if k in ('num_leaves', 'min_data_in_leaf') else float(v))
             for k, v in U_BEST_HP.items()}
print(f"\n  best on validation: {U_BEST_HP}  (valid MASE {U_HP.iloc[0]['valid_MASE']:.4f})")
print(f"  spread across the grid: {U_HP['valid_MASE'].min():.4f} .. "
      f"{U_HP['valid_MASE'].max():.4f}  "
      f"({(U_HP['valid_MASE'].max()/U_HP['valid_MASE'].min()-1)*100:.1f}% range)")
U_HP.to_csv(U_OUT / 'tables' / 'u11_hyperparameter_search.csv', index=False)

In [ ]:
# ─── Cell U11.2 — Multi-seed robustness ────────────────────────────────────
print("=" * 78)
print("  U11.2 — MULTI-SEED ROBUSTNESS")
print("=" * 78)

U_SEEDS = [42, 123, 456, 789, 2024, 31337, 777, 8080]
print(f"  seeds: {U_SEEDS}  ({len(U_SEEDS)} runs)")
print("  The seed is never chosen on test. Every seed is reported.\n")

_seed_pred, _seed_rows = {}, []
_t_start = _time.perf_counter()
for _seed in U_SEEDS:
    _frames = []
    for _t in UP.TARGETS:
        for _h in UP.HORIZONS:
            _feat = UF.build_features(U_CAL, _t, _h)
            _feat = u_attach_base(_feat, U_BASE_WIDE[('Persistence', _t, _h)])
            _feat['base_minus_lag1'] = _feat['base'] - _feat['lag_1']
            _fc = UF.feature_columns(_feat)
            _sc = U_SCORER.scale_for(_feat[UP.STATE].to_numpy(), _t)
            _fit, _yh, _te, _yv, _va = UT.fit_predict_normalized(
                'lgb', _feat, _fc, _sc, base=_feat['base'].to_numpy(float),
                params=U_BEST_HP, seed=_seed)
            # same validation-calibrated shrinkage as Section U4
            _vb = _va['base'].to_numpy(float)
            _vr = _yv - _vb
            _bl, _bs = 0.0, _np.inf
            for _lam in _np.linspace(0, 1, 11):
                _s = U_SCORER.score_group(
                    u_frame(_va, _vb + _lam * _vr, _t, _h), _t)['MASE']
                if _s < _bs:
                    _bl, _bs = float(_lam), float(_s)
            _tb = _te['base'].to_numpy(float)
            _frames.append(u_frame(_te, _tb + _bl * (_yh - _tb), _t, _h))
            del _feat
    _p = _pd.concat(_frames, ignore_index=True)
    _seed_pred[_seed] = _p
    _hd = U_SCORER.headline(_p.assign(model=f'seed{_seed}'))
    _seed_rows.append({'seed': _seed,
                       'MASE': float(_hd.iloc[0]['MASE']),
                       'RMSSE': float(_hd.iloc[0]['RMSSE']),
                       'MAE': float(_hd.iloc[0]['MAE']),
                       'sMAPE': float(_hd.iloc[0]['sMAPE']),
                       'avg_MAPE_floored_6t': float(_hd.iloc[0]['avg_MAPE_floored_6t'])})
    print(f"    seed {_seed:<6} MASE={_seed_rows[-1]['MASE']:.4f}   "
          f"({_time.perf_counter()-_t_start:.0f}s)")

U_SEED_TBL = _pd.DataFrame(_seed_rows)

In [ ]:
# ─── Cell U11.3 — Seed summary, CI, and the persistence comparison ─────────
print("=" * 78)
print("  U11.3 — SEED SUMMARY")
print("=" * 78)

_pers_head = U_SCORER.headline(U_REG.preds['Persistence'].assign(model='Persistence'))
U_PERS_MASE = float(_pers_head.iloc[0]['MASE'])

print()
print(U_SEED_TBL.round(4).to_string(index=False))

_n = len(U_SEED_TBL)
_summary = {}
for _m in ['MASE', 'RMSSE', 'MAE', 'sMAPE', 'avg_MAPE_floored_6t']:
    _v = U_SEED_TBL[_m].to_numpy(float)
    _mean, _sd = float(_v.mean()), float(_v.std(ddof=1))
    _se = _sd / _np.sqrt(_n)
    from scipy import stats as _sst
    _tcrit = float(_sst.t.ppf(0.975, df=_n - 1))
    _summary[_m] = {'mean': _mean, 'std': _sd,
                    'ci_lo': _mean - _tcrit * _se, 'ci_hi': _mean + _tcrit * _se,
                    'min': float(_v.min()), 'max': float(_v.max())}

print(f"\n  Across {_n} seeds:\n")
print(f"    {'metric':<24} {'mean':>9} {'std':>9} {'95% CI':>22} {'min':>9} {'max':>9}")
for _m, _s in _summary.items():
    print(f"    {_m:<24} {_s['mean']:>9.4f} {_s['std']:>9.4f} "
          f"[{_s['ci_lo']:>9.4f}, {_s['ci_hi']:>8.4f}] {_s['min']:>9.4f} {_s['max']:>9.4f}")

_mm = _summary['MASE']
print(f"\n  Persistence MASE = {U_PERS_MASE:.4f}")
_all_beat = bool(U_SEED_TBL['MASE'].max() < U_PERS_MASE)
_ci_beat = bool(_mm['ci_hi'] < U_PERS_MASE)
print(f"    every seed beats persistence            : {_all_beat}")
print(f"    upper 95% CI bound below persistence    : {_ci_beat}")
print(f"    mean improvement                        : "
      f"{(U_PERS_MASE - _mm['mean'])/U_PERS_MASE*100:+.2f}%")
if _ci_beat:
    print("\n  => the improvement survives re-seeding: the entire confidence interval")
    print("     sits below the persistence benchmark.")
else:
    print("\n  => the improvement does NOT clearly survive re-seeding; the confidence")
    print("     interval overlaps the persistence benchmark and the result must be")
    print("     reported as inconclusive.")

# The seed-averaged forecast is itself a legitimate (and more stable) model.
_avg = None
for _s, _p in _seed_pred.items():
    _q = _p.set_index([UP.DATE, UP.STATE, 'target', 'horizon'])
    _avg = _q[['y', 'yhat']] if _avg is None else _avg.join(
        _q[['yhat']], rsuffix=f'_{_s}', how='inner')
_yh_cols = [c for c in _avg.columns if c.startswith('yhat')]
_avg['yhat_mean'] = _avg[_yh_cols].mean(axis=1)
_seed_avg = _avg.reset_index()[[UP.DATE, UP.STATE, 'target', 'horizon', 'y', 'yhat_mean']] \
    .rename(columns={'yhat_mean': 'yhat'})
U_REG.add('Hybrid_Pers_LGBM_seedavg', _seed_avg, section='U11',
          notes=f'mean forecast over {len(U_SEEDS)} seeds, tuned hyperparameters')
U_REG.checkpoint()

_sa = U_SCORER.headline(_seed_avg.assign(model='seedavg'))
print(f"\n  Seed-averaged forecast MASE = {float(_sa.iloc[0]['MASE']):.4f} "
      f"(vs single-seed mean {_mm['mean']:.4f})")

U_SEED_TBL.to_csv(U_OUT / 'tables' / 'u11_multiseed.csv', index=False)
_pd.DataFrame(_summary).T.to_csv(U_OUT / 'tables' / 'u11_seed_summary.csv')

_fig, _ax = _plt.subplots(figsize=(7.5, 4.4))
_ax.axhline(U_PERS_MASE, color='crimson', ls='--', lw=1.5, label='Persistence')
_ax.axhspan(_mm['ci_lo'], _mm['ci_hi'], color='#1a7f37', alpha=0.18, label='95% CI of mean')
_ax.axhline(_mm['mean'], color='#1a7f37', lw=1.6, label='seed mean')
_ax.plot(range(len(U_SEED_TBL)), U_SEED_TBL['MASE'], 'o', color='#0b4f6c', label='individual seeds')
_ax.set_xticks(range(len(U_SEED_TBL)))
_ax.set_xticklabels(U_SEED_TBL['seed'], fontsize=8)
_ax.set_xlabel('seed'); _ax.set_ylabel('MASE (observable targets)')
_ax.set_title('Multi-seed robustness of the leading model', fontweight='bold', fontsize=11)
_ax.legend(fontsize=8); _ax.grid(alpha=0.3)
_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U11_multiseed.png', dpi=150, bbox_inches='tight')
print(f"\n  saved -> {U_OUT / 'figures' / 'U11_multiseed.png'}")
_plt.show()

---
## Section U12 — Regime and State Error Analysis

A single headline number hides where a model actually works. This section
breaks the final candidates down two ways: by operating regime, and across all
18 states.

**On the regimes.** The shipped stratifiers are unusable on TEST — `coal_critical`
is identically zero across all 7,614 test rows, and `regime` is constant
("Post-Shift(2022-25)"). Section U1.6 therefore redefined six overlapping
regimes from **per-state TRAIN quantiles**: Normal, Coal-critical, Monsoon,
High-renewable, High-price, High-stress. Their test coverage is reported
alongside every result, because a regime with few test rows cannot support a
confident claim.

In [ ]:
# ─── Cell U12.1 — Final candidates ─────────────────────────────────────────
print("=" * 78)
print("  U12.1 — CANDIDATES FOR THE BREAKDOWN")
print("=" * 78)

_all_head = U_SCORER.headline(U_REG.stacked(list(U_REG.preds)))
_top = [m for m in _all_head['model'].tolist()
        if m not in ('Persistence', 'SeasonalNaive')][:3]
U_FINAL_CANDIDATES = list(dict.fromkeys(_top + ['Persistence', 'SeasonalNaive']))
if 'PatchTST+MCAG(replica)' in U_REG.preds:
    U_FINAL_CANDIDATES.append('PatchTST+MCAG(replica)')
print(f"  {U_FINAL_CANDIDATES}")

_fin_stack = U_REG.stacked(U_FINAL_CANDIDATES)
print(f"  aligned evaluation points: {len(_fin_stack)//len(U_FINAL_CANDIDATES):,} per model")

In [ ]:
# ─── Cell U12.2 — Error by regime ──────────────────────────────────────────
print("=" * 78)
print("  U12.2 — ERROR BY REGIME  (observable targets)")
print("=" * 78)

_reg_tbl = U_SCORER.by_regime(_fin_stack, U_REGIME_MAP, UP.REGIME_COLS,
                              targets=UP.OBSERVABLE_TARGETS)
_reg_tbl['regime_label'] = _reg_tbl['regime'].map(UP.REGIME_LABELS)

for _metric in ['MASE', 'MAE', 'RMSE', 'MAPE']:
    print(f"\n  --- {_metric} ---")
    _p = _reg_tbl.pivot_table(index='regime_label', columns='model', values=_metric)
    _p = _p.reindex([UP.REGIME_LABELS[c] for c in UP.REGIME_COLS])
    print(_p[[c for c in U_FINAL_CANDIDATES if c in _p.columns]].round(4).to_string())

print("\n  Test-row support per regime (both observable targets, h=1..3):\n")
_supp = (_reg_tbl[_reg_tbl.model == U_FINAL_CANDIDATES[0]]
         .groupby('regime_label')['n_obs'].sum().reindex(
             [UP.REGIME_LABELS[c] for c in UP.REGIME_COLS]))
for _r, _n in _supp.items():
    _flag = '' if _n > 500 else '   <- thin support, treat with caution'
    print(f"    {_r:<18} {int(_n):>7,} rows{_flag}")
_reg_tbl.to_csv(U_OUT / 'tables' / 'u12_error_by_regime.csv', index=False)

In [ ]:
# ─── Cell U12.3 — Error by state (all 18) ──────────────────────────────────
print("=" * 78)
print("  U12.3 — ERROR BY STATE  (all 18, observable targets)")
print("=" * 78)

_st_tbl = U_SCORER.by_state(_fin_stack, targets=UP.OBSERVABLE_TARGETS)
_scale_med = U_DF[U_DF['_split'] == 'train'].groupby(UP.STATE)['total_generation_mwh'].median()

for _t in UP.OBSERVABLE_TARGETS:
    print(f"\n  --- {_t}: MASE by state ---\n")
    _p = _st_tbl[_st_tbl.target == _t].pivot(index='state_name', columns='model',
                                             values='MASE')
    _p = _p[[c for c in U_FINAL_CANDIDATES if c in _p.columns]]
    _p.insert(0, 'train_median_gen', _scale_med)
    _best_model = U_FINAL_CANDIDATES[0]
    if _best_model in _p.columns and 'Persistence' in _p.columns:
        _p['delta_vs_pers_%'] = (_p[_best_model] - _p['Persistence']) / _p['Persistence'] * 100
    print(_p.sort_values('train_median_gen').round(4).to_string())
    if 'delta_vs_pers_%' in _p.columns:
        _won = int((_p['delta_vs_pers_%'] < 0).sum())
        print(f"\n    {_best_model} beats persistence in {_won}/{len(_p)} states "
              f"(median change {_p['delta_vs_pers_%'].median():+.2f}%)")

_st_tbl.to_csv(U_OUT / 'tables' / 'u12_error_by_state.csv', index=False)

In [ ]:
# ─── Cell U12.4 — Regime/state visualisation ───────────────────────────────
_fig, _axes = _plt.subplots(1, 2, figsize=(16, 5))

_ax = _axes[0]
_p = _reg_tbl.pivot_table(index='regime_label', columns='model', values='MASE')
_p = _p.reindex([UP.REGIME_LABELS[c] for c in UP.REGIME_COLS])
_p = _p[[c for c in U_FINAL_CANDIDATES if c in _p.columns]]
_p.plot(kind='bar', ax=_ax, width=0.8)
_ax.set_ylabel('MASE'); _ax.set_xlabel('')
_ax.set_title('(a) Error by operating regime', fontweight='bold', fontsize=11)
_ax.legend(fontsize=7); _ax.grid(axis='y', alpha=0.3)
_plt.setp(_ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)

_ax = _axes[1]
_p2 = _st_tbl[_st_tbl.target == 'total_generation_mwh'].pivot(
    index='state_name', columns='model', values='MASE')
_bm = U_FINAL_CANDIDATES[0]
if _bm in _p2.columns and 'Persistence' in _p2.columns:
    _d = ((_p2[_bm] - _p2['Persistence']) / _p2['Persistence'] * 100).sort_values()
    _ax.barh(_d.index, _d.values,
             color=['#1a7f37' if v < 0 else '#c0392b' for v in _d.values])
    _ax.axvline(0, color='black', lw=1)
    _ax.set_xlabel(f'MASE change of {_bm} vs persistence (%)')
    _ax.set_title('(b) Per-state effect, generation', fontweight='bold', fontsize=11)
    _ax.tick_params(axis='y', labelsize=7.5); _ax.grid(axis='x', alpha=0.3)
_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U12_regime_state.png', dpi=150, bbox_inches='tight')
print(f"  saved -> {U_OUT / 'figures' / 'U12_regime_state.png'}")
_plt.show()

---
## Section U13 — Statistical Significance

A lower MAPE is not evidence. This section tests whether the differences
survive an explicit hypothesis test and a resampling interval.

**Diebold–Mariano.** Applied to the *daily cross-state mean loss differential*,
not to the pooled state-day vector. Pooling would treat 18 contemporaneous,
strongly cross-correlated observations as independent draws and inflate the
test statistic — with ~7,500 "observations" almost any difference becomes
"significant". Averaging across states first leaves ~420 genuinely sequential
daily observations, which is the honest sample size. Both absolute-error and
squared-error loss are reported.

**Moving-block bootstrap.** 95% confidence intervals for the MASE difference,
resampling 14-day blocks so short-range serial dependence is preserved.

Comparisons are run against all three references the brief requires:
persistence, seasonal naive, and PatchTST+MCAG.

In [ ]:
# ─── Cell U13.1 — Diebold-Mariano tests ────────────────────────────────────
print("=" * 78)
print("  U13.1 — DIEBOLD-MARIANO TESTS")
print("=" * 78)

U_REFERENCES = [r for r in ['Persistence', 'SeasonalNaive', 'PatchTST+MCAG(replica)']
                if r in U_REG.preds]
_challengers = [m for m in U_FINAL_CANDIDATES if m not in U_REFERENCES][:3]
print(f"  challengers: {_challengers}")
print(f"  references : {U_REFERENCES}\n")

_dm_rows = []
for _ch in _challengers:
    for _ref in U_REFERENCES:
        _pair = U_REG.get(_ch, _ref)
        _a = _pair[_pair.model == _ch]
        _b = _pair[_pair.model == _ref]
        for _t in UP.OBSERVABLE_TARGETS:
            for _loss in ['mae', 'mse']:
                _r = UE.dm_panel(_a, _b, _t, loss=_loss)
                _dm_rows.append({'challenger': _ch, 'reference': _ref, 'target': _t,
                                 'loss': _loss, 'n_days': _r['n'],
                                 'DM': _r['DM'], 'p_value': _r['p'],
                                 'better': (_ch if _r.get('better') == 'a' else _ref)
                                 if _r.get('better') else None,
                                 'significant_5pct': (_r['p'] < 0.05)
                                 if _np.isfinite(_r['p']) else False})
U_DM = _pd.DataFrame(_dm_rows)

for _ch in _challengers:
    print(f"\n  --- {_ch} ---")
    _s = U_DM[U_DM.challenger == _ch]
    print(f"    {'reference':<26} {'target':<22} {'loss':<5} {'DM':>8} {'p':>8}  verdict")
    for _, _r in _s.iterrows():
        _v = ('no significant difference' if not _r['significant_5pct']
              else f"{_r['better']} better (p<0.05)")
        print(f"    {_r['reference']:<26} {_r['target']:<22} {_r['loss']:<5} "
              f"{_r['DM']:>8.3f} {_r['p_value']:>8.4f}  {_v}")

U_DM.to_csv(U_OUT / 'tables' / 'u13_diebold_mariano.csv', index=False)

In [ ]:
# ─── Cell U13.2 — Block bootstrap confidence intervals ─────────────────────
print("=" * 78)
print("  U13.2 — MOVING-BLOCK BOOTSTRAP (14-day blocks, 2000 resamples)")
print("=" * 78)
print("  CI is for the DIFFERENCE in daily mean absolute error (challenger minus")
print("  reference). An interval strictly below zero means the challenger is better")
print("  at the 5% level under resampling.\n")

_bs_rows = []
for _ch in _challengers:
    for _ref in U_REFERENCES:
        _pair = U_REG.get(_ch, _ref)
        for _t in UP.OBSERVABLE_TARGETS:
            _a = _pair[(_pair.model == _ch) & (_pair.target == _t)]
            _b = _pair[(_pair.model == _ref) & (_pair.target == _t)]
            _m = _a.merge(_b, on=[UP.DATE, UP.STATE, 'target', 'horizon'],
                          suffixes=('_a', '_b'))
            _d = (_pd.DataFrame({
                UP.DATE: _m[UP.DATE],
                'd': _np.abs(_m['y_a'] - _m['yhat_a']) - _np.abs(_m['y_a'] - _m['yhat_b'])})
                .groupby(UP.DATE)['d'].mean().sort_index().to_numpy())
            _pt, _lo, _hi = UE.block_bootstrap_ci(_d, block=14, n_boot=2000, seed=42)
            _bs_rows.append({'challenger': _ch, 'reference': _ref, 'target': _t,
                             'mean_daily_MAE_diff': _pt, 'ci_lo': _lo, 'ci_hi': _hi,
                             'excludes_zero': bool(_hi < 0 or _lo > 0),
                             'direction': 'challenger better' if _pt < 0 else 'reference better'})
U_BOOT = _pd.DataFrame(_bs_rows)
print(U_BOOT.round(4).to_string(index=False))
U_BOOT.to_csv(U_OUT / 'tables' / 'u13_block_bootstrap.csv', index=False)

print("\n  Summary of the significance evidence:")
for _ch in _challengers:
    for _ref in U_REFERENCES:
        _dm_sig = U_DM[(U_DM.challenger == _ch) & (U_DM.reference == _ref)]
        _n_sig = int((_dm_sig['significant_5pct'] & (_dm_sig['better'] == _ch)).sum())
        _bt = U_BOOT[(U_BOOT.challenger == _ch) & (U_BOOT.reference == _ref)]
        _n_ci = int((_bt['excludes_zero'] & (_bt['mean_daily_MAE_diff'] < 0)).sum())
        print(f"    {_ch:<26} vs {_ref:<24} DM favours it in {_n_sig}/{len(_dm_sig)} tests; "
              f"bootstrap CI excludes 0 in {_n_ci}/{len(_bt)} targets")

---
## Section U14 — Final Leaderboard

One table, every model from Sections U2–U11, all scored on the **intersection
of every model's evaluation points** so no row enjoys an easier subset.

* **Primary ranking: MASE** (macro-averaged across the 18 states, pooled over
  h = 1..3, observable targets). MASE < 1 beats a seasonal-naive forecast.
* Secondary: RMSSE, RMSE, MAE, MAPE, sMAPE.
* `DM vs Persistence` and `DM vs PatchTST+MCAG` carry the test statistic and
  p-value from Section U13.
* `95% CI` is the moving-block bootstrap interval for MASE.

**MAPE is a percentage error, not an accuracy.** A 28.20% MAPE is not "71.8%
accurate", and no column in this table should ever be relabelled that way.

In [ ]:
# ─── Cell U14.1 — Assemble the leaderboard ─────────────────────────────────
print("=" * 78)
print("  U14.1 — FINAL LEADERBOARD")
print("=" * 78)

U_LB_MODELS = [m for m in U_REG.preds if m != 'LightGBM-rawlevel']
_lb_stack = U_REG.stacked(U_LB_MODELS)
_n_common = len(_lb_stack) // max(len(U_LB_MODELS), 1)
print(f"  {len(U_LB_MODELS)} models on {_n_common:,} common evaluation points each")
print(f"  (intersection across every model; the widest single-model grid was "
      f"{max(len(p) for p in U_REG.preds.values()):,} rows)\n")

_lb = U_SCORER.headline(_lb_stack)

# bootstrap CI for each model's MASE, and DM against the two anchors
_ci, _dmp, _dmm = {}, {}, {}
_ref_p = U_REG.preds.get('Persistence')
_ref_m = U_REG.preds.get('PatchTST+MCAG(replica)')
for _m in U_LB_MODELS:
    _p = U_REG.preds[_m]
    _d = []
    for _t in UP.OBSERVABLE_TARGETS:
        _s = _p[_p.target == _t]
        _sc = _np.array([U_SCORER.scales.get((st, _t), 1.0) for st in _s[UP.STATE]])
        _d.append(_pd.DataFrame({UP.DATE: _s[UP.DATE],
                                 'e': _np.abs(_s['y'] - _s['yhat']) / _sc})
                  .groupby(UP.DATE)['e'].mean())
    _daily = _pd.concat(_d, axis=1).mean(axis=1).sort_index().to_numpy()
    _pt, _lo, _hi = UE.block_bootstrap_ci(_daily, block=14, n_boot=1000, seed=7)
    _ci[_m] = (_lo, _hi)
    for _ref, _store in [(_ref_p, _dmp), (_ref_m, _dmm)]:
        if _ref is None or _m in ('Persistence', 'PatchTST+MCAG(replica)'):
            _store[_m] = (_np.nan, _np.nan)
            continue
        _rs = [UE.dm_panel(_p, _ref, _t, loss='mae') for _t in UP.OBSERVABLE_TARGETS]
        _rs = [r for r in _rs if _np.isfinite(r['DM'])]
        _store[_m] = ((float(_np.mean([r['DM'] for r in _rs])),
                       float(_np.max([r['p'] for r in _rs]))) if _rs else (_np.nan, _np.nan))

_lb['CI_lo'] = _lb['model'].map(lambda m: _ci[m][0])
_lb['CI_hi'] = _lb['model'].map(lambda m: _ci[m][1])
_lb['DM_vs_Persistence'] = _lb['model'].map(lambda m: _dmp.get(m, (_np.nan,))[0])
_lb['p_vs_Persistence'] = _lb['model'].map(lambda m: _dmp.get(m, (_np.nan, _np.nan))[1])
_lb['DM_vs_PatchTST_MCAG'] = _lb['model'].map(lambda m: _dmm.get(m, (_np.nan,))[0])
_lb['p_vs_PatchTST_MCAG'] = _lb['model'].map(lambda m: _dmm.get(m, (_np.nan, _np.nan))[1])
_lb = _lb.sort_values('MASE').reset_index(drop=True)
_lb.insert(0, 'Rank', _np.arange(1, len(_lb) + 1))
U_LEADERBOARD = _lb

print(f"  {'#':>3} {'Model':<28} {'MASE':>7} {'sMAPE':>7} {'MAE':>9} {'RMSE':>9} "
      f"{'RMSSE':>7} {'MAPE':>7} {'95% CI (MASE units)':>22}")
print("  " + "-" * 112)
for _, _r in _lb.iterrows():
    print(f"  {int(_r['Rank']):>3} {_r['model']:<28} {_r['MASE']:>7.4f} {_r['sMAPE']:>7.3f} "
          f"{_r['MAE']:>9.3f} {_r['RMSE']:>9.3f} {_r['RMSSE']:>7.4f} {_r['MAPE']:>7.3f} "
          f"[{_r['CI_lo']:>8.4f},{_r['CI_hi']:>8.4f}]")

print("\n  Significance columns (mean DM over the observable targets, worst-case p;")
print("  DM < 0 means the row beats the reference):\n")
print(f"  {'#':>3} {'Model':<28} {'DM vs Persist':>14} {'p':>8} {'DM vs PatchTST':>15} {'p':>8}")
print("  " + "-" * 80)
for _, _r in _lb.iterrows():
    print(f"  {int(_r['Rank']):>3} {_r['model']:<28} {_r['DM_vs_Persistence']:>14.3f} "
          f"{_r['p_vs_Persistence']:>8.4f} {_r['DM_vs_PatchTST_MCAG']:>15.3f} "
          f"{_r['p_vs_PatchTST_MCAG']:>8.4f}")

U_LEADERBOARD.to_csv(U_OUT / 'tables' / 'u14_final_leaderboard.csv', index=False)
U_REG.dump_meta('u14_model_registry.json')
print(f"\n  saved -> {U_OUT / 'tables' / 'u14_final_leaderboard.csv'}")

In [ ]:
# ─── Cell U14.2 — Leaderboard figure ───────────────────────────────────────
_fig, _axes = _plt.subplots(1, 2, figsize=(16, max(5, 0.32 * len(U_LEADERBOARD))))

_ax = _axes[0]
_d = U_LEADERBOARD.iloc[::-1]
_pers = float(U_LEADERBOARD.set_index('model').loc['Persistence', 'MASE'])
_cols = ['#c0392b' if m == 'Persistence' else
         ('#1a7f37' if v < _pers else '#7f8c8d')
         for m, v in zip(_d['model'], _d['MASE'])]
_ax.barh(range(len(_d)), _d['MASE'], color=_cols)
_ax.errorbar(_d['MASE'], range(len(_d)),
             xerr=[_d['MASE'] - _d['CI_lo'], _d['CI_hi'] - _d['MASE']],
             fmt='none', ecolor='black', elinewidth=0.8, capsize=2)
_ax.axvline(_pers, color='crimson', ls='--', lw=1.4)
_ax.set_yticks(range(len(_d)))
_ax.set_yticklabels(_d['model'], fontsize=7)
_ax.set_xlabel('MASE (observable targets, macro across states)')
_ax.set_title('(a) Final leaderboard — lower is better', fontweight='bold', fontsize=11)
_ax.grid(axis='x', alpha=0.3)

_ax = _axes[1]
_top6 = U_LEADERBOARD.head(6)['model'].tolist()
_tth = U_SCORER.by_target_horizon(U_REG.stacked(list(dict.fromkeys(_top6 + ['Persistence']))))
for _m in dict.fromkeys(_top6 + ['Persistence']):
    _s = (_tth[(_tth.model == _m) & (_tth.target.isin(UP.OBSERVABLE_TARGETS))]
          .groupby('horizon')['MASE'].mean())
    _ax.plot(_s.index, _s.values, marker='o', lw=1.8,
             ls='--' if _m == 'Persistence' else '-', label=_m)
_ax.set_xlabel('forecast horizon (days ahead)'); _ax.set_ylabel('MASE')
_ax.set_xticks(UP.HORIZONS)
_ax.set_title('(b) Top models by horizon', fontweight='bold', fontsize=11)
_ax.legend(fontsize=7.5); _ax.grid(alpha=0.3)

_plt.tight_layout()
_plt.savefig(U_OUT / 'figures' / 'U14_leaderboard.png', dpi=150, bbox_inches='tight')
print(f"  saved -> {U_OUT / 'figures' / 'U14_leaderboard.png'}")
_plt.show()

---
## Section U15 — Final Research Conclusion

Generated from the measured results above, not written by hand. The verdicts
are computed from the leaderboard, the Section U13 tests and the Section U11
seed study, so the text cannot drift from the numbers.

In [ ]:
# ─── Cell U15.1 — Automatic verdict ────────────────────────────────────────
print("=" * 78)
print("  U15.1 — AUTOMATIC RESULT SUMMARY")
print("=" * 78)

_LB = U_LEADERBOARD.set_index('model')
U_PERSISTENCE_MASE = float(_LB.loc['Persistence', 'MASE'])
_non_trivial = [m for m in U_LEADERBOARD['model']
                if m not in ('Persistence', 'SeasonalNaive', 'RollingMean', 'Drift')]
U_BEST_MODEL = U_LEADERBOARD[U_LEADERBOARD.model.isin(_non_trivial)].iloc[0]['model']
U_BEST_MASE = float(_LB.loc[U_BEST_MODEL, 'MASE'])
U_OVERALL_BEST = str(U_LEADERBOARD.iloc[0]['model'])

_PUB_MAPE, _PUB_MASE = 28.20, 3.4748
_replica = 'PatchTST+MCAG(replica)'
_rep_mase = float(_LB.loc[_replica, 'MASE']) if _replica in _LB.index else _np.nan

print(f"""
  CURRENT RE-FUSED (published, Part I, untouched)
      model                : PatchTST + MCAG
      avg MAPE (6 targets) : {_PUB_MAPE:.2f}%
      headline MASE        : {_PUB_MASE:.4f}   (pooled scaling, observable targets)
      note                 : scored pooled while its baselines were scored per-state
                             then macro-averaged, so the published 3.4748 and the
                             published naive 0.494 were never on the same scale.

  SAME ARCHITECTURE, REPRODUCED ON THIS GRID
      model                : {_replica}
      MASE (macro)         : {_rep_mase:.4f}
      -> confirms the published pattern is reproducible and is driven by the input
         specification (no lag of the target), not by the transformer.

  NEW BEST MODEL
      model                : {U_BEST_MODEL}
      MASE (macro)         : {U_BEST_MASE:.4f}
      95% CI               : [{float(_LB.loc[U_BEST_MODEL,'CI_lo']):.4f}, {float(_LB.loc[U_BEST_MODEL,'CI_hi']):.4f}]
      sMAPE                : {float(_LB.loc[U_BEST_MODEL,'sMAPE']):.3f}
      MAE / RMSE           : {float(_LB.loc[U_BEST_MODEL,'MAE']):.3f} / {float(_LB.loc[U_BEST_MODEL,'RMSE']):.3f}
      avg MAPE (6 targets) : {float(_LB.loc[U_BEST_MODEL,'avg_MAPE_floored_6t']):.2f}%   (floored denominator, as in Part I)

  PERSISTENCE BENCHMARK
      MASE (macro)         : {U_PERSISTENCE_MASE:.4f}
      95% CI               : [{float(_LB.loc['Persistence','CI_lo']):.4f}, {float(_LB.loc['Persistence','CI_hi']):.4f}]

  IMPROVEMENT
      vs persistence       : {(U_PERSISTENCE_MASE - U_BEST_MASE)/U_PERSISTENCE_MASE*100:+.2f}%
      vs PatchTST+MCAG     : {((_rep_mase - U_BEST_MASE)/_rep_mase*100) if _np.isfinite(_rep_mase) else float('nan'):+.2f}%   (both on this grid)
""")

# --- the four verdicts, computed ------------------------------------------
_beats_pers = bool(U_BEST_MASE < U_PERSISTENCE_MASE)
_beats_mcag = bool(_np.isfinite(_rep_mase) and U_BEST_MASE < _rep_mase)

_dm_p = U_DM[(U_DM.challenger == U_BEST_MODEL) & (U_DM.reference == 'Persistence')] \
    if 'U_DM' in dir() else _pd.DataFrame()
_sig_pers = bool(len(_dm_p) and (_dm_p['significant_5pct'] & (_dm_p['better'] == U_BEST_MODEL)).all())
_bt = U_BOOT[(U_BOOT.challenger == U_BEST_MODEL) & (U_BOOT.reference == 'Persistence')] \
    if 'U_BOOT' in dir() else _pd.DataFrame()
_ci_pers = bool(len(_bt) and (_bt['excludes_zero'] & (_bt['mean_daily_MAE_diff'] < 0)).all())

_seeds_ok = None
if 'U_SEED_TBL' in dir() and len(U_SEED_TBL):
    from scipy import stats as _sst
    _v = U_SEED_TBL['MASE'].to_numpy(float)
    _n = len(_v); _sd = _v.std(ddof=1)
    _hi = _v.mean() + _sst.t.ppf(0.975, _n - 1) * _sd / _np.sqrt(_n)
    _seeds_ok = bool(_hi < U_PERSISTENCE_MASE)
    _seed_txt = (f"{_n} seeds, MASE {_v.mean():.4f} +/- {_sd:.4f}; "
                 f"upper 95% CI {_hi:.4f} vs persistence {U_PERSISTENCE_MASE:.4f}")
else:
    _seed_txt = "not run"

print("  " + "=" * 74)
print("  VERDICTS")
print("  " + "=" * 74)
for _q, _a in [
    ("Does the new best model beat PERSISTENCE?", _beats_pers),
    ("Does it beat PatchTST+MCAG (same grid)?", _beats_mcag),
    ("Is the gain vs persistence statistically significant (DM, 5%)?", _sig_pers),
    ("Does the bootstrap CI vs persistence exclude zero?", _ci_pers),
    ("Does it survive multi-seed re-runs?", _seeds_ok),
]:
    _mark = {True: 'YES', False: 'NO', None: 'NOT TESTED'}[_a]
    print(f"    {_q:<62} {_mark}")

In [ ]:
# ─── Cell U15.2 — Paper-ready conclusion ───────────────────────────────────
_state_tbl = U_SCORER.by_state(U_REG.get(U_BEST_MODEL, 'Persistence'),
                               targets=UP.OBSERVABLE_TARGETS)
_ps = _state_tbl.pivot_table(index='state_name', columns='model', values='MASE')
_n_states_won = int((_ps[U_BEST_MODEL] < _ps['Persistence']).sum()) \
    if U_BEST_MODEL in _ps.columns else 0

_by_h = U_SCORER.by_target_horizon(U_REG.get(U_BEST_MODEL, 'Persistence'))
_h_tbl = _by_h[_by_h.target.isin(UP.OBSERVABLE_TARGETS)].pivot_table(
    index='horizon', columns='model', values='MASE')

_lines = []
_lines.append("=" * 78)
_lines.append("  U15.2 — CONCLUSION (paper-ready)")
_lines.append("=" * 78)
_lines.append("")
_lines.append("We re-examined day-ahead forecasting on the RE-FUSED-Alpha panel (18 Indian")
_lines.append("states, 47,286 state-day observations, chronological test period")
_lines.append("2024-01-01 to 2025-04-22) using an isolated pipeline with a mechanical")
_lines.append("leakage proof: every observation from the test boundary onward was replaced")
_lines.append("with noise and the pre-boundary features verified bit-identical across")
_lines.append(f"{18} target-horizon configurations and 712,152 row-comparisons.")
_lines.append("")
_lines.append("The published PatchTST+MCAG result (28.20% average MAPE, MASE 3.4748) is")
_lines.append("reproducible, and its weakness is attributable to the input specification")
_lines.append("rather than the architecture: the original feature list contains no lag of")
_lines.append("total_generation_mwh, so the model was asked to predict the level of a")
_lines.append("series it was never shown. Two further measurement issues were identified —")
_lines.append("the published model MASE was computed with pooled scaling while its")
_lines.append("baselines were macro-averaged across states, and the validation split was a")
_lines.append("state holdout rather than a time holdout.")
_lines.append("")
_lines.append(f"Persistence is a strong benchmark on this panel (MASE {U_PERSISTENCE_MASE:.4f}),")
_lines.append("beating ARIMA, ETS, seasonal-naive and rolling-mean alternatives. Market")
_lines.append("price in particular is near-trivially persistent, which the original audit")
_lines.append("attributes to the 87.3% imputation rate smoothing the series.")
_lines.append("")
if _beats_pers:
    _lines.append(f"The strongest model found is {U_BEST_MODEL}, at MASE")
    _lines.append(f"{U_BEST_MASE:.4f} against persistence at {U_PERSISTENCE_MASE:.4f} — an improvement of")
    _lines.append(f"{(U_PERSISTENCE_MASE-U_BEST_MASE)/U_PERSISTENCE_MASE*100:.2f}%. It wins in {_n_states_won} of 18 states.")
    _lines.append("The gain comes from two corrections rather than from model capacity:")
    _lines.append("forecasting the residual from a persistence base (which removes the")
    _lines.append("train-to-test level shift) and normalising each state's target by its own")
    _lines.append("MASE denominator (which aligns the training loss with the macro-averaged")
    _lines.append("metric on a panel spanning three orders of magnitude).")
    _lines.append("")
    _lines.append("Statistically: " + (
        "Diebold-Mariano tests on the daily cross-state loss differential reject equal"
        if _sig_pers else
        "Diebold-Mariano tests do NOT consistently reject equal"))
    _lines.append("accuracy against persistence at the 5% level" +
                  (", and the moving-block" if _sig_pers else ", and the moving-block"))
    _lines.append(("bootstrap interval excludes zero." if _ci_pers
                   else "bootstrap interval for the error difference includes zero."))
    if _seeds_ok is True:
        _lines.append("The improvement survives re-seeding: the 95% confidence interval over")
        _lines.append(f"{len(U_SEED_TBL)} seeds lies entirely below the persistence benchmark.")
    elif _seeds_ok is False:
        _lines.append("The improvement does NOT survive re-seeding: the seed confidence")
        _lines.append("interval overlaps the persistence benchmark, so it should be reported")
        _lines.append("as inconclusive rather than as an improvement.")
else:
    _lines.append("No model tested beats persistence on the headline metric. The strongest")
    _lines.append(f"learned model, {U_BEST_MODEL}, reaches MASE {U_BEST_MASE:.4f}")
    _lines.append(f"against persistence at {U_PERSISTENCE_MASE:.4f}. We report this as the result:")
    _lines.append("on this panel, at a 1-3 day horizon, a random-walk forecast is not")
    _lines.append("improved on by tree ensembles, N-BEATS, N-HiTS, TSMixer, carbon-gated")
    _lines.append("transformers or their ensembles.")
_lines.append("")
_lines.append("Limitations. (i) 87.3% of state-day rows carry at least one imputed value;")
_lines.append("the smoothing this induces is the most likely reason persistence is so hard")
_lines.append("to beat, and it caps what any model can demonstrate here. (ii) The mean")
_lines.append("market price roughly doubles between train and test, so 85.6% of test rows")
_lines.append("sit above the per-state training 90th percentile — a covariate shift no")
_lines.append("fixed-range learner handles well. (iii) The test period contains a 54-day")
_lines.append("data outage (2024-12-11 to 2025-02-02) which removes window-based models'")
_lines.append("coverage immediately afterwards. (iv) Horizons are limited to 1-3 days to")
_lines.append("match the original PRED=3 setting; longer horizons, where persistence decays")
_lines.append("fastest, are where a learned model would most plausibly win and are left to")
_lines.append("future work. (v) No time-series foundation model was evaluated, because")
_lines.append("installing one would have altered the pinned torch build that produced the")
_lines.append("Part I results.")
_lines.append("")
_lines.append("Reporting note: MAPE is a percentage error and is never restated as an")
_lines.append("accuracy. A 28.20% MAPE does not correspond to 71.8% accuracy.")
_lines.append("")

_txt = "\n".join(_lines)
print(_txt)
(U_OUT / 'outputs').mkdir(parents=True, exist_ok=True)
(U_OUT / 'outputs' / 'u15_conclusion.txt').write_text(_txt, encoding='utf-8')

_summary = {
    'published': {'model': 'PatchTST+MCAG', 'avg_mape': _PUB_MAPE, 'mase': _PUB_MASE},
    'replica_same_grid': {'model': _replica,
                          'mase': None if not _np.isfinite(_rep_mase) else _rep_mase},
    'new_best': {'model': U_BEST_MODEL, 'mase': U_BEST_MASE,
                 'ci': [float(_LB.loc[U_BEST_MODEL, 'CI_lo']),
                        float(_LB.loc[U_BEST_MODEL, 'CI_hi'])]},
    'persistence': {'mase': U_PERSISTENCE_MASE},
    'verdicts': {'beats_persistence': _beats_pers, 'beats_patchtst_mcag': _beats_mcag,
                 'dm_significant_vs_persistence': _sig_pers,
                 'bootstrap_ci_excludes_zero': _ci_pers,
                 'survives_multiseed': _seeds_ok},
    'multiseed': _seed_txt,
    'n_states_won_vs_persistence': _n_states_won,
    'evaluation': {'test_period': ['2024-01-01', '2025-04-22'], 'horizons': UP.HORIZONS,
                   'targets': UP.TARGETS, 'headline_targets': UP.OBSERVABLE_TARGETS,
                   'primary_metric': 'MASE (macro across 18 states, m=7)'},
}
(U_OUT / 'tables' / 'u15_final_summary.json').write_text(
    _json.dumps(_summary, indent=2, default=str), encoding='utf-8')
print(f"  saved -> {U_OUT / 'outputs' / 'u15_conclusion.txt'}")
print(f"  saved -> {U_OUT / 'tables' / 'u15_final_summary.json'}")